<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9_multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 9 (multiclass) — normalized, bridge-consistent hNPE–hNDE training

This is an **alternative** to `Exercise_9_Hybrid_NPE_NDE.ipynb`; the original exercise is unchanged.  It keeps the same Gaussian simulator and sample-only construction, while making three upgrades deliberately visible:

1. every structured ratio head has the same approximately 3.16M-parameter residual architecture;
2. every proposal is an explicit equal-weight mixture of independently trained flows, and vector flows use learned LU mixing between spline couplings;
3. the controlled Part-I comparison is **CE**, **CE–normalization**, and **CE–normalization–bridge**.  The last objective is used for the explicit-nuisance experiment and all downstream applications.

Here “CE” is shorthand for the common classification objective: multiclass cross entropy plus the direct balanced proper score for the $a$-head.  No analytic simulator density, handcrafted density residual, or density-aware input is used in training.  Known design densities and learned proposal-mixture densities enter only the normalization/bridge objectives and reconstruction formulae.

The final section demonstrates the additional capabilities in Table 2 of the hybrid-NSBI paper: corrected likelihood generation, absolute evidence, posterior-predictive generation, selection integrals, and systematic-prior updates.  Analytic Gaussian quantities are used only as held-out validation truth.


In [ ]:
# ========================================================================
# Google Colab setup — safe to re-run; a no-op outside Colab.
# ========================================================================
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = True

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)
    run(sys.executable, "-m", "pip", "install", "-q", "nflows==0.14")
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
else:
    candidates = [Path.cwd(), Path.cwd() / "workshops" / "ml4hep_tifr_colab"]
    TUTORIAL_DIR = next(
        (candidate for candidate in candidates if (candidate / "utils_hnpe.py").exists()),
        None,
    )
    if TUTORIAL_DIR is None:
        raise FileNotFoundError(
            "Run from the repository root or workshops/ml4hep_tifr_colab."
        )
    if str(TUTORIAL_DIR.resolve()) not in sys.path:
        sys.path.insert(0, str(TUTORIAL_DIR.resolve()))


## What is actually being tested?

A balanced $K$-class classifier identifies pairwise density ratios through logit differences.  Finite networks do not automatically make those ratios normalized or mutually Bayes-compatible, so we compare three nested objectives on exactly the same sampled rows:

$$
\begin{aligned}
\mathcal L_{\rm CE} &= \mathcal L_{\rm MC}
 +\lambda_a(\mathcal L_a-\log 2),\\
\mathcal L_{\rm CE-Z} &= \mathcal L_{\rm CE}+\lambda_Z\mathcal L_Z,\\
\mathcal L_{\rm CE-Z-B} &= \mathcal L_{\rm CE-Z}+\lambda_B\mathcal L_B.
\end{aligned}
$$

$\mathcal L_Z$ is the **normalization loss**: it estimates conditional moments such as $Z_P(x)=\mathbb E_{q_P}[r_P]$ and $Z_L(\theta)=\mathbb E_{q_L}[r_L]$ from independent A/B proposal banks, and penalizes $(Z-1)^2$ with an unbiased cross product.  It fixes the absolute mass of each corrected conditional density; CE alone only fixes ratios at the population optimum.

$\mathcal L_B$ is an **evidence-invariance bridge**.  Posterior and likelihood routes imply the same evidence, so the reconstructed log evidence must not vary with the integration parameter at fixed $x$.  The training bridge uses the raw implied evidence and an unbiased within-anchor sample variance.  It deliberately does not differentiate through noisy nested estimates of $\log Z$.  The separately trained normalization loss supplies those constants; fresh, fully normalized bridge curves are reserved for validation.

This bridge is not an algebraic logit cycle and is not a simulator-closure test: it also contains known design densities and the learned proposal-mixture densities.  It can enforce mutual consistency even if both learned routes are wrong, so simulator closure, PIT/coverage, tail diagnostics, and independent normalization checks remain mandatory.


In [ ]:
import copy
import gc
import hashlib
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from scipy.special import logsumexp
from scipy.stats import multivariate_normal, norm, wasserstein_distance
from torch.utils.data import DataLoader, TensorDataset

from utils_dual_hnde import importance_tail_summary
from utils_hnpe import (
    sample_spline_flow,
    sample_spline_flow_ensemble,
    scalar_spline_flow_cdf,
    scalar_spline_flow_icdf,
    scalar_spline_flow_ensemble_cdf,
    scalar_spline_flow_ensemble_icdf,
    spline_flow_log_prob,
    spline_flow_ensemble_log_prob,
    train_spline_flow_ensemble,
)
from utils_plotting import export_standalone_figure_script


SEED = 19092026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


def set_torch_seed(seed):
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def _flow_log_prob(flow_pack, target, *, context=None, batch_size=65_536):
    """Density of one flow or the arithmetic equal-weight flow mixture."""
    if isinstance(flow_pack, (list, tuple)):
        return spline_flow_ensemble_log_prob(
            flow_pack, target, context=context, batch_size=batch_size
        )
    return spline_flow_log_prob(
        flow_pack, target, context=context, batch_size=batch_size
    )


# This committed configuration is the paper-quality full run.  FAST_MODE is
# a practical first Colab pass; SMOKE_MODE checks only the complete code path.
SMOKE_MODE = False
FAST_MODE = False
LOAD_IF_AVAILABLE = False

if SMOKE_MODE:
    RUN_TAG = "smoke"
    N_FLOW, N_CLASS = 2_000, 3_000
    SCALAR_FLOW_EPOCHS, VECTOR_FLOW_EPOCHS = 2, 2
    CLASS_EPOCHS, ENSEMBLE_SIZE, FLOW_ENSEMBLE_SIZE = 5, 1, 1
    N_NORM_TRAIN_GROUPS, N_NORM_TRAIN_INNER, N_NORM_TRAIN_BANKS = 32, 8, 1
    N_NORM_VALID_GROUPS, N_NORM_VALID_INNER, N_NORM_VALID_BANKS = 32, 16, 1
    N_DIAGNOSTIC_REFERENCE = 64
    N_FLOW_AUDIT_SAMPLES = 128
    N_BRIDGE_TRAIN_GROUPS, N_BRIDGE_TRAIN_INNER = 16, 4
    N_BRIDGE_VALID_GROUPS, N_BRIDGE_VALID_INNER = 16, 8
elif FAST_MODE:
    RUN_TAG = "fast"
    N_FLOW, N_CLASS = 35_000, 70_000
    SCALAR_FLOW_EPOCHS, VECTOR_FLOW_EPOCHS = 12, 18
    CLASS_EPOCHS, ENSEMBLE_SIZE, FLOW_ENSEMBLE_SIZE = 28, 2, 2
    N_NORM_TRAIN_GROUPS, N_NORM_TRAIN_INNER, N_NORM_TRAIN_BANKS = 512, 32, 4
    N_NORM_VALID_GROUPS, N_NORM_VALID_INNER, N_NORM_VALID_BANKS = 128, 64, 2
    N_DIAGNOSTIC_REFERENCE = 256
    N_FLOW_AUDIT_SAMPLES = 1_024
    N_BRIDGE_TRAIN_GROUPS, N_BRIDGE_TRAIN_INNER = 256, 16
    N_BRIDGE_VALID_GROUPS, N_BRIDGE_VALID_INNER = 128, 24
else:
    RUN_TAG = "full"
    N_FLOW, N_CLASS = 500_000, 750_000
    SCALAR_FLOW_EPOCHS, VECTOR_FLOW_EPOCHS = 50, 80
    CLASS_EPOCHS, ENSEMBLE_SIZE, FLOW_ENSEMBLE_SIZE = 70, 4, 4
    N_NORM_TRAIN_GROUPS, N_NORM_TRAIN_INNER, N_NORM_TRAIN_BANKS = 4_096, 64, 8
    N_NORM_VALID_GROUPS, N_NORM_VALID_INNER, N_NORM_VALID_BANKS = 512, 128, 4
    N_DIAGNOSTIC_REFERENCE = 768
    N_FLOW_AUDIT_SAMPLES = 8_192
    N_BRIDGE_TRAIN_GROUPS, N_BRIDGE_TRAIN_INNER = 2_048, 24
    N_BRIDGE_VALID_GROUPS, N_BRIDGE_VALID_INNER = 512, 48

N_GRID_REFERENCE = 48 if SMOKE_MODE else (128 if FAST_MODE else 256)
N_CALIBRATION_CONTEXTS = 20 if SMOKE_MODE else (200 if FAST_MODE else 2_000)
N_CALIBRATION_SAMPLES = 128 if SMOKE_MODE else (1_024 if FAST_MODE else 2_048)
N_FOUR_CALIBRATION_CONTEXTS = 8 if SMOKE_MODE else (100 if FAST_MODE else 500)
N_FOUR_CALIBRATION_MU = 16 if SMOKE_MODE else (128 if FAST_MODE else 256)
N_FOUR_CALIBRATION_ALPHA = 8 if SMOKE_MODE else (16 if FAST_MODE else 32)

MODEL_DIR = Path("models_exercise9_multiclass_v4_bridge_ensemble") / RUN_TAG
FIGURE_SCRIPT_DIR = Path("exercise9_multiclass_v4_figures_scripts") / RUN_TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_SCRIPT_DIR.mkdir(parents=True, exist_ok=True)


# These vector flows are completely sample-trained.  The first two coupling
# layers are learned full-support conditional affine maps; the following RQS
# stack has a much wider interval and enough bins to preserve central detail.
VECTOR_FLOW_MODEL_CONFIG = {
    "architecture_version": "full_support_affine_lu_rqs_v3",
    "n_unbounded_affine_layers": 2,
    "affine_hidden_features": 256,
    "affine_hidden_layers": 2,
    "use_layer_permutations": False,
    "linear_mixing": "lu",
    "n_coupling_layers": 10,
    "hidden_features": 512,
    "hidden_layers": 4,
    "spline_num_bins": 80,
    "spline_tail_bound": 25.0,
    "activation": "silu",
    "dropout_probability": 0.0,
    "identity_initialization": True,
}
VECTOR_FLOW_TRAINING_CONFIG = {
    "batch_size": 1024,
    "n_epochs": VECTOR_FLOW_EPOCHS,
    "learning_rate": 3.0e-4,
    "min_learning_rate": 1.0e-6,
    "lr_scheduler_factor": 0.5,
    "lr_scheduler_patience": 3,
    "validation_fraction": 0.2,
    "patience": 15,
    "gradient_clip": 1.0,
    "weight_decay": 1.0e-6,
    "retain_initial_model": True,
}


# Scalar targets cannot benefit from coordinate mixing, so q_P and q_N use
# a deeper stack of monotone conditional RQS transforms.  Their arithmetic
# mixture supplies the multimodal flexibility; every member keeps the exact
# inverse/forward numerical preflight below.
SCALAR_FLOW_MODEL_CONFIG = {
    "architecture_version": "stacked_scalar_rqs_mixture_v4",
    "n_coupling_layers": 4,
    "hidden_features": 256,
    "hidden_layers": 4,
    "spline_num_bins": 48,
    "spline_tail_bound": 18.0,
    "activation": "silu",
    "dropout_probability": 0.0,
    "identity_initialization": True,
}
SCALAR_FLOW_TRAINING_CONFIG = {
    "batch_size": 1024,
    "n_epochs": SCALAR_FLOW_EPOCHS,
    "learning_rate": 3.0e-4,
    "min_learning_rate": 1.0e-7,
    "validation_fraction": 0.2,
    "patience": 10,
    "weight_decay": 1.0e-5,
    "gradient_clip": 1.0,
    "lr_scheduler_patience": 2,
    "retain_initial_model": True,
}


# Every ratio head now matches the successful binary-corrector scale: width
# 512 and six residual blocks.  Exact counts differ slightly with input width
# and are printed before training.
CLASS_MODEL_CONFIG = {
    "a_hidden_features": 512 if not SMOKE_MODE else 128,
    "a_residual_blocks": 6 if not SMOKE_MODE else 2,
    "aux_hidden_features": 512 if not SMOKE_MODE else 128,
    "aux_residual_blocks": 6 if not SMOKE_MODE else 2,
    "dropout_probability": 0.0,
    "log_ratio_bound": 20.0,
    "architecture_version": "structured_equal_capacity_bridge_heads_v4",
}
CLASS_TRAINING_CONFIG = {
    "batch_size_groups": 1024 if not SMOKE_MODE else 256,
    "constraint_batch_groups": 16 if not SMOKE_MODE else 8,
    "constraint_validation_chunk_groups": 16,
    "validation_batch_groups": 512 if not SMOKE_MODE else 128,
    "n_epochs": CLASS_EPOCHS,
    "learning_rate": 1.0e-3,
    "learning_rate_factors": (1.0, 0.30, 0.10, 0.03, 0.01),
    "direct_a_loss_weight": 1.0,
    "direct_a_objective_version": "balanced_pairwise_bce_v1",
    "weight_decay": 1.0e-5,
    "validation_fraction": 0.2,
    "patience": max(6, CLASS_EPOCHS // 10),
    "minimum_improvement": 1.0e-6,
    "gradient_clip": 5.0,
    "warmup_epochs": min(2, max(1, CLASS_EPOCHS // 3)),
    "ramp_epochs": min(5, max(1, CLASS_EPOCHS // 2)),
    "bridge_warmup_epochs": min(8, max(2, CLASS_EPOCHS // 4)),
    "bridge_ramp_epochs": min(5, max(1, CLASS_EPOCHS // 2)),
}
LAMBDA_NORM_3 = 0.20
LAMBDA_BRIDGE_3 = 0.05
LAMBDA_NORM_4 = 0.15
LAMBDA_BRIDGE_4 = 0.03
THREE_DIRECT_LABEL = "CE"
THREE_NORMALIZED_LABEL = "CE + normalization"
THREE_BRIDGE_LABEL = "CE + normalization + bridge"

SIMULATOR_SIGMA = np.array([0.95, 0.38, 0.30], dtype=float)
X_OBS = np.array([0.40, 1.35, 0.12], dtype=float)


def export_exercise9_multiclass_figure(fig, script_name):
    path = export_standalone_figure_script(
        fig, script_name=script_name, output_dir=FIGURE_SCRIPT_DIR
    )
    png_path = FIGURE_SCRIPT_DIR / f"{Path(script_name).stem}.png"
    pdf_path = FIGURE_SCRIPT_DIR / f"{Path(script_name).stem}.pdf"
    fig.savefig(png_path, dpi=220, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print("Exported:", path, png_path, pdf_path)
    return path


print(
    f"mode={RUN_TAG}, N_flow={N_FLOW:,}, N_class={N_CLASS:,}, "
    f"classifier ensemble={ENSEMBLE_SIZE}, flow mixture={FLOW_ENSEMBLE_SIZE}"
)


## The unchanged Gaussian simulator and exact validation densities

We retain Exercise 9 exactly:

$$
\begin{aligned}
  x_1&\sim\mathcal N(\mu+0.8\alpha,0.95^2),\\
  x_2&\sim\mathcal N(0.72\mu^2-0.4\alpha,0.38^2),\\
  x_3&\sim\mathcal N(0.8\cos\mu+0.3\alpha,0.30^2),
\end{aligned}
$$

with $\rho_\mu=0.9\mathcal N(0,1.5^2)+0.1\mathcal N(0,4^2)$ and
$\rho_\alpha=0.9\mathcal N(0,1^2)+0.1\mathcal N(0,3^2)$.

When $\alpha$ is hidden, it can be integrated analytically for validation.  Writing
$b(\mu)=(\mu,0.72\mu^2,0.8\cos\mu)$ and $v=(0.8,-0.4,0.3)$,

$$
  p_m(x\mid\mu)=0.9\,\mathcal N(x;b,\Sigma_\epsilon+vv^T)
  +0.1\,\mathcal N(x;b,\Sigma_\epsilon+9vv^T).
$$

Neither this expression nor the explicit analytic likelihood is passed to a network.


In [ ]:
def _mixture_logpdf(values, core_sigma, broad_sigma):
    values = np.asarray(values, dtype=float)
    return logsumexp(
        np.stack([
            np.log(0.9) + norm.logpdf(values, 0.0, core_sigma),
            np.log(0.1) + norm.logpdf(values, 0.0, broad_sigma),
        ]),
        axis=0,
    )

def design_mu_logpdf(mu):
    return _mixture_logpdf(mu, 1.5, 4.0)

def design_alpha_logpdf(alpha):
    return _mixture_logpdf(alpha, 1.0, 3.0)

def design_logpdf(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    return design_mu_logpdf(theta[:, 0]) + design_alpha_logpdf(theta[:, 1])

def _sample_mixture(n, core_sigma, broad_sigma, rng):
    broad = rng.random(int(n)) < 0.1
    sigma = np.where(broad, broad_sigma, core_sigma)
    return rng.normal(0.0, sigma)

def sample_mu(n, rng):
    return _sample_mixture(n, 1.5, 4.0, rng).astype(np.float32)

def sample_alpha(n, rng):
    return _sample_mixture(n, 1.0, 3.0, rng).astype(np.float32)

def sample_design(n, rng):
    return np.column_stack([sample_mu(n, rng), sample_alpha(n, rng)]).astype(
        np.float32
    )

def simulator_base_mean(mu):
    mu = np.asarray(mu, dtype=float).ravel()
    return np.column_stack([mu, 0.72 * mu**2, 0.8 * np.cos(mu)])

def simulator_mean(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    mu, alpha = theta[:, 0], theta[:, 1]
    return simulator_base_mean(mu) + alpha[:, None] * np.array([0.8, -0.4, 0.3])

def simulate(theta, rng):
    mean = simulator_mean(theta)
    return (mean + rng.normal(size=mean.shape) * SIMULATOR_SIGMA).astype(
        np.float32
    )

def log_likelihood(x, theta):
    """Analytic truth used only in validation cells."""
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    x = np.atleast_2d(np.asarray(x, dtype=float))
    if len(x) == 1 and len(theta) > 1:
        x = np.repeat(x, len(theta), axis=0)
    if len(x) != len(theta):
        raise ValueError("x and theta must have matching rows or one x row.")
    return np.sum(norm.logpdf(x, simulator_mean(theta), SIMULATOR_SIGMA), axis=1)

_NOISE_COV = np.diag(SIMULATOR_SIGMA**2)
_ALPHA_LOADING = np.array([0.8, -0.4, 0.3])
_MARGINAL_COVS = [
    _NOISE_COV + sigma_alpha**2 * np.outer(_ALPHA_LOADING, _ALPHA_LOADING)
    for sigma_alpha in (1.0, 3.0)
]

def marginal_log_likelihood(x, mu):
    """Exact p_m(x|mu) after integrating the design nuisance mixture."""
    mu = np.asarray(mu, dtype=float).ravel()
    x = np.atleast_2d(np.asarray(x, dtype=float))
    if len(x) == 1 and len(mu) > 1:
        x = np.repeat(x, len(mu), axis=0)
    if len(x) != len(mu):
        raise ValueError("x and mu must have matching rows or one x row.")
    residual = x - simulator_base_mean(mu)
    components = [
        np.log(weight)
        + np.atleast_1d(
            multivariate_normal.logpdf(residual, mean=np.zeros(3), cov=cov)
        )
        for weight, cov in zip((0.9, 0.1), _MARGINAL_COVS)
    ]
    return np.atleast_1d(logsumexp(np.stack(components), axis=0))

def normalize_log_curve(log_density, grid):
    log_density = np.asarray(log_density, dtype=float)
    shift = float(np.max(log_density))
    density = np.exp(log_density - shift)
    integral = np.trapezoid(density, grid)
    return density / integral, shift + np.log(integral)

def normalize_log_surface(log_density, x_grid, y_grid):
    log_density = np.asarray(log_density, dtype=float)
    shift = float(np.max(log_density))
    density = np.exp(log_density - shift)
    integral = np.trapezoid(np.trapezoid(density, y_grid, axis=1), x_grid)
    return density / integral, shift + np.log(integral)

def integrated_absolute_error(reference, estimate, grid):
    return float(np.trapezoid(np.abs(reference - estimate), grid))

def js_distance_discrete(reference, estimate, floor=1.0e-15):
    reference = np.asarray(reference, dtype=float).ravel() + floor
    estimate = np.asarray(estimate, dtype=float).ravel() + floor
    reference /= reference.sum()
    estimate /= estimate.sum()
    middle = 0.5 * (reference + estimate)
    divergence = 0.5 * np.sum(reference * np.log(reference / middle))
    divergence += 0.5 * np.sum(estimate * np.log(estimate / middle))
    return float(np.sqrt(max(0.0, divergence)))

print("Observed x:", X_OBS)
print("Truth check, log p_m(x_obs|mu=1.2):", marginal_log_likelihood(X_OBS, [1.2])[0])



def sample_vector_flow_design(n_samples, rng, score_columns):
    """Empirically stratify vector-flow contexts without evaluating a density."""

    n_samples = int(n_samples)
    pool = sample_design(max(n_samples, 100_000), rng)
    selected_values = pool[:, list(score_columns)]
    center = np.median(selected_values, axis=0)
    mad = 1.4826 * np.median(np.abs(selected_values - center), axis=0)
    scale = np.where(mad > 1.0e-6, mad, selected_values.std(axis=0))
    scale = np.where(scale > 1.0e-6, scale, 1.0)
    score = np.max(np.abs((selected_values - center) / scale), axis=1)
    q90, q99 = np.quantile(score, [0.90, 0.99])
    strata = [score < q90, (score >= q90) & (score < q99), score >= q99]
    fractions = np.array([0.71, 0.21, 0.08])
    counts = np.floor(fractions * n_samples).astype(int)
    counts[0] += n_samples - counts.sum()
    pieces = []
    for mask, count in zip(strata, counts):
        candidates = np.flatnonzero(mask)
        pieces.append(
            pool[rng.choice(candidates, size=int(count), replace=count > len(candidates))]
        )
    result = np.concatenate(pieces, axis=0)
    return result[rng.permutation(len(result))].astype(np.float32)


def sample_tail_enriched_design(n_samples, rng, tail_fraction=0.25):
    """Sample constraint anchors with extra empirical tail coverage.

    The target condition is Z(c)=1 for every c, so changing only the anchor
    weighting does not change that population solution.  Selection uses
    sampled parameter ranks; no simulator density is evaluated.
    """

    n_samples = int(n_samples)
    n_tail = min(n_samples, max(1, int(round(tail_fraction * n_samples))))
    bulk = sample_design(n_samples - n_tail, rng)
    pool = sample_design(max(1_024, 12 * n_tail), rng)
    center = np.median(pool, axis=0)
    mad = 1.4826 * np.median(np.abs(pool - center), axis=0)
    scale = np.where(mad > 1.0e-6, mad, pool.std(axis=0))
    scale = np.where(scale > 1.0e-6, scale, 1.0)
    score = np.max(np.abs((pool - center) / scale), axis=1)
    tail_pool = np.flatnonzero(score >= np.quantile(score, 0.90))
    selected = rng.choice(tail_pool, size=n_tail, replace=len(tail_pool) < n_tail)
    combined = np.concatenate([bulk, pool[selected]], axis=0)
    return combined[rng.permutation(len(combined))].astype(np.float32)


## Structured sample-only ratio trainer

Classifier inputs are robustly standardized sampled coordinates followed by the invertible compression $t\mapsto\operatorname{asinh}(t)$.  No density value, analytic score, or handcrafted residual coordinate is supplied to a head.

All $a,b,c$ heads now use width 512 and six pre-normalized residual blocks—about 3.16M parameters each.  Every head is zero-initialized and has a gently bounded log-ratio output; saturation is reported.  For classes $(S,P,L)$,

$$[s_S,s_P,s_L]=[a,0,a-c],$$

and $a=\log(S/P)$ receives direct balanced BCE.  For $(S,N,P,L)$,

$$[s_S,s_N,s_P,s_L]=[a+b,a,0,a+b-c].$$

Here $a=\log(N/P)$ excludes $\alpha$ by architecture and receives direct $N/P$ BCE, while $\log(S/P)=a+b$ remains the exact structured composition.

One AdamW optimizer is preserved through five progressively smaller learning-rate phases.  Normalization ramps in after a short CE warmup; the bridge ramps in later, after the mass constraint has stabilized.  Checkpoint selection evaluates every active term on disjoint held-out class groups and constraint banks.


In [ ]:
class PreNormResidualBlock(nn.Module):
    def __init__(self, width, dropout_probability=0.0):
        super().__init__()
        self.norm = nn.LayerNorm(int(width))
        self.linear_one = nn.Linear(int(width), int(width))
        self.linear_two = nn.Linear(int(width), int(width))
        self.dropout = nn.Dropout(float(dropout_probability))

    def forward(self, values):
        update = self.linear_one(self.norm(values))
        update = F.silu(update)
        update = self.dropout(update)
        update = self.linear_two(update)
        return values + update / math.sqrt(2.0)


class RatioHead(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_features,
        residual_blocks,
        dropout_probability,
    ):
        super().__init__()
        self.input_layer = nn.Linear(int(input_dim), int(hidden_features))
        self.blocks = nn.ModuleList([
            PreNormResidualBlock(hidden_features, dropout_probability)
            for _ in range(int(residual_blocks))
        ])
        self.final_norm = nn.LayerNorm(int(hidden_features))
        self.output_layer = nn.Linear(int(hidden_features), 1)
        nn.init.zeros_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)

    def forward(self, values):
        hidden = F.silu(self.input_layer(values))
        for block in self.blocks:
            hidden = block(hidden)
        return self.output_layer(F.silu(self.final_norm(hidden)))[:, 0]


class StructuredRatioNetwork(nn.Module):
    def __init__(
        self,
        input_dim,
        n_classes,
        a_hidden_features=512,
        a_residual_blocks=6,
        aux_hidden_features=512,
        aux_residual_blocks=6,
        dropout_probability=0.0,
        log_ratio_bound=20.0,
        architecture_version=None,
    ):
        super().__init__()
        del architecture_version
        self.input_dim = int(input_dim)
        self.n_classes = int(n_classes)
        self.log_ratio_bound = float(log_ratio_bound)
        if (self.n_classes, self.input_dim) not in {(3, 4), (4, 5)}:
            raise ValueError("Expected (n_classes,input_dim)=(3,4) or (4,5).")
        a_common = dict(
            hidden_features=a_hidden_features,
            residual_blocks=a_residual_blocks,
            dropout_probability=dropout_probability,
        )
        auxiliary_common = dict(
            hidden_features=aux_hidden_features,
            residual_blocks=aux_residual_blocks,
            dropout_probability=dropout_probability,
        )
        a_input_dim = self.input_dim if self.n_classes == 3 else 4
        self.a_head = RatioHead(a_input_dim, **a_common)
        self.c_head = RatioHead(self.input_dim, **auxiliary_common)
        self.b_head = None
        if self.n_classes == 4:
            self.b_head = RatioHead(self.input_dim, **auxiliary_common)

    def raw_ratio_heads(self, values):
        if self.n_classes == 3:
            a_values = values
        else:
            # [mu, alpha, x1, x2, x3] -> [mu, x1, x2, x3].
            a_values = values[:, [0, 2, 3, 4]]
        heads = {"a": self.a_head(a_values), "c": self.c_head(values)}
        if self.b_head is not None:
            heads["b"] = self.b_head(values)
        return heads

    def bounded_ratio_heads(self, values):
        raw = self.raw_ratio_heads(values)
        bound = self.log_ratio_bound
        if bound <= 0.0:
            return raw
        return {name: bound * torch.tanh(value / bound) for name, value in raw.items()}

    def forward(self, values):
        heads = self.bounded_ratio_heads(values)
        a, c = heads["a"], heads["c"]
        zeros = torch.zeros_like(a)
        if self.n_classes == 3:
            return torch.stack([a, zeros, a - c], dim=1)
        b = heads["b"]
        return torch.stack([a + b, a, zeros, a + b - c], dim=1)


def _assert_finite(name, values, ndim=None):
    values = np.asarray(values)
    if ndim is not None and values.ndim != ndim:
        raise ValueError(f"{name} must have ndim={ndim}; got {values.shape}.")
    if not np.isfinite(values).all():
        raise ValueError(f"{name} contains non-finite values.")
    return values

def _install_nflows_rqs_float64_retry():
    """Retry only a failed float32 inverse-RQS kernel in float64.

    For a monotone rational-quadratic spline the inverse discriminant is
    non-negative analytically.  nflows 0.14 evaluates it in float32, where a
    nearly double root can acquire a tiny negative value by cancellation.  We
    retry the *same* spline tensors in float64: no row is dropped, clipped, or
    resampled.  The original float64 assertion remains the hard guard.
    """
    import functools
    import importlib
    import inspect
    from importlib.metadata import version
    import warnings

    nflows_version = version("nflows")
    if nflows_version != "0.14":
        raise RuntimeError(
            "This audited numerical guard requires nflows==0.14; "
            f"found {nflows_version}."
        )
    module = importlib.import_module(
        "nflows.transforms.splines.rational_quadratic"
    )
    original = module.rational_quadratic_spline
    if getattr(original, "_exercise9_float64_retry", False):
        return original
    signature = inspect.signature(original)

    @functools.wraps(original)
    def guarded(*args, **kwargs):
        inputs_fast = kwargs.get("inputs", args[0] if args else None)
        inverse_fast = kwargs.get(
            "inverse", args[4] if len(args) > 4 else False
        )
        if (
            inverse_fast
            and torch.is_tensor(inputs_fast)
            and inputs_fast.dtype == torch.float32
        ):
            guarded._float32_inverse_call_count += 1
            guarded._float32_inverse_values += int(inputs_fast.numel())
        try:
            return original(*args, **kwargs)
        except AssertionError as error32:
            bound = signature.bind(*args, **kwargs)
            inputs = bound.arguments["inputs"]
            inverse = bound.arguments.get(
                "inverse", signature.parameters["inverse"].default
            )
            if not inverse or inputs.dtype != torch.float32:
                raise

            floating = [
                value for value in (*args, *kwargs.values())
                if torch.is_tensor(value) and value.is_floating_point()
            ]
            if any(not bool(torch.isfinite(value).all()) for value in floating):
                raise FloatingPointError(
                    "Non-finite tensor reached the inverse RQS; refusing the "
                    "precision retry."
                ) from error32

            def to_float64(value):
                if torch.is_tensor(value) and value.is_floating_point():
                    return value.to(dtype=torch.float64)
                return value

            try:
                outputs64, logdet64 = original(
                    *(to_float64(value) for value in args),
                    **{
                        name: to_float64(value)
                        for name, value in kwargs.items()
                    },
                )
            except AssertionError as error64:
                raise RuntimeError(
                    "The inverse-RQS discriminant also failed in float64. "
                    "Refusing to clip or resample; retrain this flow."
                ) from error64
            if not (
                bool(torch.isfinite(outputs64).all())
                and bool(torch.isfinite(logdet64).all())
            ):
                raise FloatingPointError(
                    "The float64 inverse-RQS retry returned non-finite values."
                ) from error32

            guarded._float64_retry_count += 1
            guarded._float64_retry_values += int(inputs.numel())
            if guarded._float64_retry_count == 1:
                warnings.warn(
                    "nflows float32 inverse-RQS cancellation: retrying the "
                    "same spline call in float64.",
                    RuntimeWarning,
                    stacklevel=2,
                )
            outputs = outputs64.to(dtype=inputs.dtype)
            logdet = logdet64.to(dtype=inputs.dtype)
            if not (
                bool(torch.isfinite(outputs).all())
                and bool(torch.isfinite(logdet).all())
            ):
                raise FloatingPointError(
                    "Casting the inverse-RQS retry back to float32 "
                    "produced non-finite values."
                ) from error32
            return outputs, logdet

    guarded._exercise9_float64_retry = True
    guarded._float64_retry_count = 0
    guarded._float64_retry_values = 0
    guarded._float32_inverse_call_count = 0
    guarded._float32_inverse_values = 0
    guarded._float32_original = original
    module.rational_quadratic_spline = guarded

    # nflows 0.14's linear-tail helper resolves the module global above.  The
    # aliases cover any direct bounded-spline call without touching package
    # source or a checkpoint.
    importlib.import_module(
        "nflows.transforms.splines"
    ).rational_quadratic_spline = guarded
    importlib.import_module(
        "nflows.transforms.autoregressive"
    ).rational_quadratic_spline = guarded
    linear_tail = importlib.import_module(
        "nflows.transforms.splines"
    ).unconstrained_rational_quadratic_spline
    if linear_tail.__globals__.get("rational_quadratic_spline") is not guarded:
        raise RuntimeError(
            "The nflows 0.14 linear-tail inverse did not bind to the "
            "audited RQS guard."
        )
    return guarded


RQS_NUMERIC_GUARD = _install_nflows_rqs_float64_retry()


def _rqs_retry_count():
    return int(getattr(RQS_NUMERIC_GUARD, "_float64_retry_count", 0))


def _rqs_inverse_call_count():
    return int(
        getattr(RQS_NUMERIC_GUARD, "_float32_inverse_call_count", 0)
    )


def _draw_conditional(flow_pack, contexts, n_samples, seed, allocation="balanced"):
    contexts = np.atleast_2d(np.asarray(contexts, dtype=np.float32))
    n_samples = int(n_samples)
    first_member = flow_pack[0] if isinstance(flow_pack, (list, tuple)) else flow_pack
    n_features = int(first_member["config"]["n_features"])
    if isinstance(flow_pack, (list, tuple)):
        draws = sample_spline_flow_ensemble(
            flow_pack,
            n_samples,
            context=contexts,
            seed=int(seed),
            batch_size=16_384,
            allocation=allocation,
        )
    else:
        set_torch_seed(seed)
        draws = sample_spline_flow(
            flow_pack, n_samples, context=contexts, batch_size=16_384
        )
    draws = np.asarray(draws, dtype=np.float32)
    if len(contexts) == 1:
        draws = draws[None, :, :]
    expected = (len(contexts), n_samples, n_features)
    if draws.shape != expected:
        raise RuntimeError(
            f"Unexpected conditional sample shape {draws.shape}; expected {expected}."
        )
    return _assert_finite("conditional flow draws", draws, ndim=3)



def _audit_context_subset(flow_pack, contexts, seed, n_contexts=2_048):
    contexts = _assert_finite("flow audit contexts", contexts, ndim=2).astype(
        np.float32
    )
    if len(contexts) < n_contexts:
        raise ValueError("The flow audit needs at least n_contexts rows.")
    member = flow_pack[0] if isinstance(flow_pack, (list, tuple)) else flow_pack
    scaler = member["context_scaler"]
    standardized = (contexts - scaler.mean) / scaler.std
    extremeness = np.max(np.abs(standardized), axis=1)
    n_tail = min(512, n_contexts // 4)
    tail_index = np.argpartition(extremeness, -n_tail)[-n_tail:]
    available = np.setdiff1d(
        np.arange(len(contexts)), tail_index, assume_unique=False
    )
    rng = np.random.default_rng(int(seed))
    typical_index = rng.choice(
        available, size=n_contexts - n_tail, replace=False
    )
    return contexts[np.concatenate([typical_index, tail_index])]


def audit_scalar_flow(flow_pack, contexts, name, seed):
    """Active inverse/forward, density, finiteness, and RNG checks."""
    if int(flow_pack["config"]["n_features"]) != 1:
        raise ValueError("audit_scalar_flow requires a scalar target flow.")
    history = flow_pack.get("history", {})
    initial_values = history.get("initial_validation", [])
    selected_values = history.get("selected_validation", [])
    if initial_values and selected_values:
        initial_validation = float(initial_values[-1])
        selected_validation = float(selected_values[-1])
        print(
            f"{name} validation NLL: identity={initial_validation:.4f}, "
            f"selected={selected_validation:.4f}."
        )
        if selected_validation >= initial_validation - 1.0e-4:
            import warnings
            warnings.warn(
                f"{name} retained its context-independent identity "
                "baseline; numerical closure can still pass, but proposal "
                "fidelity must be treated as failed until PIT/ESS checks.",
                RuntimeWarning,
                stacklevel=2,
            )
    contexts = _audit_context_subset(flow_pack, contexts, seed)
    z_grid = np.arange(-6.0, 7.0, dtype=np.float64)
    probabilities = np.tile(norm.cdf(z_grid), len(contexts))
    repeated_contexts = np.repeat(contexts, len(z_grid), axis=0)
    retry_before = _rqs_retry_count()
    inverse_calls_before = _rqs_inverse_call_count()

    quantiles = scalar_spline_flow_icdf(
        flow_pack,
        probabilities,
        context=repeated_contexts,
        batch_size=8_192,
    ).reshape(len(contexts), len(z_grid))
    if not np.isfinite(quantiles).all():
        raise FloatingPointError(f"{name}: non-finite inverse-CDF values.")
    if not np.all(np.diff(quantiles, axis=1) > 0.0):
        raise RuntimeError(f"{name}: inverse CDF is not strictly monotone.")

    recovered_probability = scalar_spline_flow_cdf(
        flow_pack,
        quantiles.reshape(-1, 1),
        context=repeated_contexts,
        batch_size=8_192,
    )
    recovered_z = norm.ppf(
        np.clip(
            recovered_probability,
            np.nextafter(0.0, 1.0),
            np.nextafter(1.0, 0.0),
        )
    ).reshape(quantiles.shape)
    z_error = np.abs(recovered_z - z_grid[None, :])
    q99_error = float(np.quantile(z_error, 0.99))
    max_error = float(np.max(z_error))
    if q99_error > 1.0e-4 or max_error > 2.0e-3:
        raise RuntimeError(
            f"{name}: inverse/forward closure failed: "
            f"q99={q99_error:.3g}, max={max_error:.3g}."
        )

    log_density = _flow_log_prob(
        flow_pack,
        quantiles.reshape(-1, 1),
        context=repeated_contexts,
        batch_size=8_192,
    )
    if not np.isfinite(log_density).all():
        raise FloatingPointError(f"{name}: non-finite density at its quantiles.")

    first = _draw_conditional(flow_pack, contexts, 2, seed + 1)
    second = _draw_conditional(flow_pack, contexts, 2, seed + 1)
    if not np.array_equal(first, second):
        raise RuntimeError(f"{name}: repeated seeded draws are not identical.")
    retry_delta = _rqs_retry_count() - retry_before
    inverse_call_delta = _rqs_inverse_call_count() - inverse_calls_before
    if retry_delta:
        raise RuntimeError(
            f"{name}: {retry_delta}/{inverse_call_delta} inverse-RQS "
            "kernel calls needed float64 in the numerical preflight. "
            "Retrain this scalar flow; reserve the fallback for a rare "
            "event in the much larger production draw."
        )
    print(
        f"{name} scalar-flow audit passed: q99 |z_back-z|={q99_error:.2e}, "
        f"max={max_error:.2e}, retried RQS kernels="
        f"{retry_delta}/{inverse_call_delta}."
    )


def _logmeanexp_torch(values, dim=-1):
    return torch.logsumexp(values, dim=dim) - math.log(values.shape[dim])


def _fit_classifier_transform(points):
    points = np.asarray(points, dtype=np.float32)
    center = np.median(points, axis=0).astype(np.float32)
    mad = np.median(np.abs(points - center), axis=0).astype(np.float32)
    robust_scale = (1.4826 * mad).astype(np.float32)
    ordinary_scale = points.std(axis=0, dtype=np.float64).astype(np.float32)
    scale = np.where(robust_scale > 1.0e-6, robust_scale, ordinary_scale)
    scale = np.where(scale > 1.0e-6, scale, 1.0).astype(np.float32)
    return center, scale


def _transform_classifier_points(points, center, scale):
    points = np.asarray(points, dtype=np.float32)
    return np.arcsinh((points - center) / scale).astype(np.float32)


def _as_constraint_banks(banks):
    if banks is None:
        return []
    return list(banks) if isinstance(banks, (list, tuple)) else [banks]


def _prepare_constraint_banks(banks, center, scale):
    prepared_banks = []
    for bundle in _as_constraint_banks(banks):
        prepared = {}
        for key, value in bundle.items():
            array = np.asarray(value, dtype=np.float32)
            if key.endswith("_points"):
                array = _transform_classifier_points(array, center, scale)
            prepared[key] = torch.as_tensor(array, dtype=torch.float32)
        prepared_banks.append(prepared)
    return prepared_banks


def _constraint_scale(epoch, config):
    warmup = int(config["warmup_epochs"])
    ramp = int(config["ramp_epochs"])
    if epoch < warmup:
        return 0.0
    return min(1.0, (epoch - warmup + 1) / max(1, ramp))


def _make_model(input_dim, n_classes):
    return StructuredRatioNetwork(
        input_dim=input_dim,
        n_classes=n_classes,
        **CLASS_MODEL_CONFIG,
    ).to(device)


def _bridge_scale(epoch, config):
    warmup = int(config["bridge_warmup_epochs"])
    ramp = int(config["bridge_ramp_epochs"])
    if epoch < warmup:
        return 0.0
    return min(1.0, (epoch - warmup + 1) / max(1, ramp))


def _multiclass_fingerprint(
    class_points,
    *,
    n_classes,
    seed_base,
    lambda_norm,
    lambda_bridge,
    constraint_train,
    constraint_validation,
    bridge_train,
    bridge_validation,
):
    digest = hashlib.sha256()
    configuration = json.dumps(
        {
            "n_classes": int(n_classes),
            "seed_base": int(seed_base),
            "ensemble_size": int(ENSEMBLE_SIZE),
            "model": CLASS_MODEL_CONFIG,
            "training": CLASS_TRAINING_CONFIG,
            "lambda_norm": float(lambda_norm),
            "lambda_bridge": float(lambda_bridge),
            "input_transform": "robust_asinh_v1",
            "normalization_estimator": "independent_cross_mass_selection_v2",
            "bridge_estimator": "raw_evidence_iid_unbiased_group_variance_v2",
        },
        sort_keys=True,
        separators=(",", ":"),
    )
    digest.update(configuration.encode("utf-8"))

    def update_array(name, value):
        array = np.ascontiguousarray(value)
        digest.update(name.encode("utf-8"))
        digest.update(str(array.shape).encode("ascii"))
        digest.update(str(array.dtype).encode("ascii"))
        digest.update(array.view(np.uint8))

    update_array("class_points", class_points)
    for collection_name, collection in [
        ("constraint_train", constraint_train),
        ("constraint_validation", constraint_validation),
        ("bridge_train", bridge_train),
        ("bridge_validation", bridge_validation),
    ]:
        banks = _as_constraint_banks(collection)
        if not banks:
            digest.update(f"{collection_name}:none".encode("utf-8"))
        for bank_index, bundle in enumerate(banks):
            for key in sorted(bundle):
                update_array(f"{collection_name}:{bank_index}:{key}", bundle[key])
    return digest.hexdigest()


def _direct_a_class_pair(n_classes):
    """Return numerator/denominator rows for the head-local a ratio."""
    if int(n_classes) == 3:
        return 0, 1, "S/P"
    if int(n_classes) == 4:
        return 1, 2, "N/P"
    raise ValueError("Direct a supervision requires three or four classes.")


def _balanced_pairwise_bce(group_logits, numerator, denominator):
    positive = group_logits[:, numerator, numerator] - group_logits[:, numerator, denominator]
    negative = group_logits[:, denominator, numerator] - group_logits[:, denominator, denominator]
    return 0.5 * (F.softplus(-positive).mean() + F.softplus(negative).mean())


def _classifier_losses(model, group_batch, n_classes):
    group_logits = model(
        group_batch.reshape(-1, group_batch.shape[-1])
    ).reshape(len(group_batch), int(n_classes), int(n_classes))
    labels = torch.arange(int(n_classes), device=group_logits.device).repeat(len(group_batch))
    ce = F.cross_entropy(group_logits.reshape(-1, int(n_classes)), labels)
    numerator, denominator, _ = _direct_a_class_pair(n_classes)
    direct_a_bce = _balanced_pairwise_bce(group_logits, numerator, denominator)
    return ce, direct_a_bce


def _validation_classifier_losses(model, points_tensor, n_classes):
    model.eval()
    ce_sum, direct_a_sum, n_rows, n_groups = 0.0, 0.0, 0, 0
    batch_groups = int(CLASS_TRAINING_CONFIG["validation_batch_groups"])
    with torch.no_grad():
        for start in range(0, len(points_tensor), batch_groups):
            batch = points_tensor[start : start + batch_groups].to(device)
            ce, direct_a_bce = _classifier_losses(model, batch, n_classes)
            ce_sum += float(ce.cpu()) * len(batch) * int(n_classes)
            direct_a_sum += float(direct_a_bce.cpu()) * len(batch)
            n_rows += len(batch) * int(n_classes)
            n_groups += len(batch)
    return ce_sum / max(1, n_rows), direct_a_sum / max(1, n_groups)


def _constraint_logits(model, bundle, key, index, n_classes):
    points = bundle[key].index_select(0, index).to(device)
    logits = model(points.reshape(-1, points.shape[-1]))
    return logits.reshape(*points.shape[:-1], int(n_classes))


def _partition_cross_mass(logits_a, logits_b, numerator, denominator):
    log_w_a = logits_a[..., numerator] - logits_a[..., denominator]
    log_w_b = logits_b[..., numerator] - logits_b[..., denominator]
    log_z_a = _logmeanexp_torch(log_w_a, dim=1)
    log_z_b = _logmeanexp_torch(log_w_b, dim=1)
    delta_a = torch.expm1(log_z_a.double())
    delta_b = torch.expm1(log_z_b.double())
    cross = (delta_a * delta_b).mean()
    pooled_monitor = (0.5 * (delta_a + delta_b)).square().mean()
    if not torch.isfinite(cross) or not torch.isfinite(pooled_monitor):
        raise FloatingPointError("Non-finite conditional mass estimate.")
    return cross.float(), pooled_monitor.float()


def _validation_mass_metrics(model, banks, loss_fn):
    if not banks or loss_fn is None:
        return 0.0, 0.0
    chunk_size = int(CLASS_TRAINING_CONFIG["constraint_validation_chunk_groups"])
    cross_sum, pooled_sum, count = 0.0, 0.0, 0
    model.eval()
    with torch.no_grad():
        for bundle in banks:
            n_groups = next(iter(bundle.values())).shape[0]
            for start in range(0, n_groups, chunk_size):
                stop = min(n_groups, start + chunk_size)
                index = torch.arange(start, stop, dtype=torch.long)
                cross, pooled = loss_fn(model, bundle, index)
                weight = stop - start
                cross_sum += float(cross.cpu()) * weight
                pooled_sum += float(pooled.cpu()) * weight
                count += weight
    return cross_sum / max(1, count), pooled_sum / max(1, count)


def _validation_bridge_metric(model, banks, loss_fn):
    if not banks or loss_fn is None:
        return 0.0
    chunk_size = int(CLASS_TRAINING_CONFIG["constraint_validation_chunk_groups"])
    value_sum, count = 0.0, 0
    model.eval()
    with torch.no_grad():
        for bundle in banks:
            n_groups = next(iter(bundle.values())).shape[0]
            for start in range(0, n_groups, chunk_size):
                stop = min(n_groups, start + chunk_size)
                index = torch.arange(start, stop, dtype=torch.long)
                value = loss_fn(model, bundle, index)
                weight = stop - start
                value_sum += float(value.cpu()) * weight
                count += weight
    return value_sum / max(1, count)


def _new_classifier_optimizer(model, learning_rate):
    return torch.optim.AdamW(
        model.parameters(),
        lr=float(learning_rate),
        betas=(0.9, 0.99),
        weight_decay=float(CLASS_TRAINING_CONFIG["weight_decay"]),
    )


def _classifier_learning_rate(epoch, n_epochs):
    factors = tuple(CLASS_TRAINING_CONFIG["learning_rate_factors"])
    stage = min(len(factors) - 1, len(factors) * int(epoch) // max(1, int(n_epochs)))
    if int(epoch) + 1 == int(n_epochs):
        stage = len(factors) - 1
    return float(CLASS_TRAINING_CONFIG["learning_rate"]) * float(factors[stage]), stage


def train_multiclass_ensemble(
    class_points,
    *,
    n_classes,
    checkpoint_dir,
    seed_base,
    constraint_train=None,
    constraint_validation=None,
    constraint_loss_fn=None,
    lambda_norm=0.0,
    bridge_train=None,
    bridge_validation=None,
    bridge_loss_fn=None,
    lambda_bridge=0.0,
):
    class_points = _assert_finite("class_points", class_points, ndim=3).astype(np.float32)
    if class_points.shape[1] != n_classes:
        raise ValueError("class_points must contain one row per class per group.")
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    fingerprint = _multiclass_fingerprint(
        class_points,
        n_classes=n_classes,
        seed_base=seed_base,
        lambda_norm=lambda_norm,
        lambda_bridge=lambda_bridge,
        constraint_train=constraint_train,
        constraint_validation=constraint_validation,
        bridge_train=bridge_train,
        bridge_validation=bridge_validation,
    )

    split_rng = np.random.default_rng(SEED + 700 + n_classes)
    order = split_rng.permutation(len(class_points))
    n_validation = max(1, int(CLASS_TRAINING_CONFIG["validation_fraction"] * len(order)))
    validation_index, training_index = order[:n_validation], order[n_validation:]
    if set(validation_index).intersection(set(training_index)):
        raise RuntimeError("Group split leakage detected.")

    training_flat = class_points[training_index].reshape(-1, class_points.shape[-1])
    center, robust_scale = _fit_classifier_transform(training_flat)
    train_tensor = torch.as_tensor(
        _transform_classifier_points(class_points[training_index], center, robust_scale),
        dtype=torch.float32,
    )
    validation_tensor = torch.as_tensor(
        _transform_classifier_points(class_points[validation_index], center, robust_scale),
        dtype=torch.float32,
    )
    train_dataset = TensorDataset(train_tensor)
    prepared_train = _prepare_constraint_banks(constraint_train, center, robust_scale)
    prepared_validation = _prepare_constraint_banks(constraint_validation, center, robust_scale)
    prepared_bridge_train = _prepare_constraint_banks(bridge_train, center, robust_scale)
    prepared_bridge_validation = _prepare_constraint_banks(bridge_validation, center, robust_scale)

    ensemble = []
    _, _, direct_a_name = _direct_a_class_pair(n_classes)
    log_two = math.log(2.0)
    direct_a_weight = float(CLASS_TRAINING_CONFIG["direct_a_loss_weight"])
    for member in range(ENSEMBLE_SIZE):
        checkpoint = checkpoint_dir / f"member_{member}.pt"
        if LOAD_IF_AVAILABLE and checkpoint.exists():
            try:
                saved = torch.load(checkpoint, map_location=device, weights_only=False)
            except TypeError:
                saved = torch.load(checkpoint, map_location=device)
            if saved.get("fingerprint") != fingerprint:
                raise RuntimeError(
                    f"Checkpoint {checkpoint} belongs to different data, architecture, loss, or constraint draws."
                )
            model = _make_model(class_points.shape[-1], n_classes)
            model.load_state_dict(saved["state_dict"])
            model.eval()
            ensemble.append({
                "model": model,
                "center": np.asarray(saved["center"], dtype=np.float32),
                "scale": np.asarray(saved["scale"], dtype=np.float32),
                "history": saved.get("history", {}),
                "checkpoint": checkpoint,
            })
            print("Loaded", checkpoint)
            continue

        member_seed = int(seed_base + member)
        set_torch_seed(member_seed)
        # Optimizer-side RNG streams live outside every precomputed-bank
        # namespace, so minibatch indices cannot share bank draw seeds.
        data_generator = torch.Generator().manual_seed(member_seed + 120_000)
        constraint_generator = torch.Generator().manual_seed(member_seed + 130_000)
        bridge_generator = torch.Generator().manual_seed(member_seed + 140_000)
        train_loader = DataLoader(
            train_dataset,
            batch_size=int(CLASS_TRAINING_CONFIG["batch_size_groups"]),
            shuffle=True,
            drop_last=False,
            generator=data_generator,
        )
        model = _make_model(class_points.shape[-1], n_classes)
        if member == 0:
            counts = {
                "a": sum(p.numel() for p in model.a_head.parameters()),
                "c": sum(p.numel() for p in model.c_head.parameters()),
            }
            if model.b_head is not None:
                counts["b"] = sum(p.numel() for p in model.b_head.parameters())
            print(f"Structured {n_classes}-class head parameter counts:", counts)
            print(
                f"Direct a proper score: {direct_a_name}, weight={direct_a_weight:.1f}; "
                f"lambda_Z={float(lambda_norm):.3g}, lambda_B={float(lambda_bridge):.3g}"
            )
        optimizer = _new_classifier_optimizer(model, CLASS_TRAINING_CONFIG["learning_rate"])
        history = {key: [] for key in [
            "train_ce", "train_a_bce", "train_mass", "train_bridge", "train_total",
            "validation_ce", "validation_a_bce", "validation_a_gain",
            "validation_mass_cross", "validation_mass_pooled", "validation_bridge",
            "validation_total", "constraint_scale", "bridge_scale", "learning_rate",
            "learning_rate_stage",
        ]}
        best_state, best_value, best_record, stale = None, math.inf, None, 0
        previous_lr_stage = None

        for epoch in range(CLASS_EPOCHS):
            learning_rate, learning_rate_stage = _classifier_learning_rate(epoch, CLASS_EPOCHS)
            for parameter_group in optimizer.param_groups:
                parameter_group["lr"] = learning_rate
            if previous_lr_stage is not None and learning_rate_stage != previous_lr_stage:
                stale = 0
            previous_lr_stage = learning_rate_stage

            model.train()
            scale_now = _constraint_scale(epoch, CLASS_TRAINING_CONFIG)
            bridge_scale_now = _bridge_scale(epoch, CLASS_TRAINING_CONFIG)
            bank = prepared_train[epoch % len(prepared_train)] if prepared_train else None
            bridge_bank = (
                prepared_bridge_train[epoch % len(prepared_bridge_train)]
                if prepared_bridge_train else None
            )
            ce_sum = direct_a_sum = mass_sum = bridge_sum = total_sum = 0.0
            n_rows, n_groups_seen = 0, 0
            for (group_batch,) in train_loader:
                group_batch = group_batch.to(device)
                ce, direct_a_bce = _classifier_losses(model, group_batch, n_classes)
                mass_loss = torch.zeros((), device=device)
                if (
                    bank is not None
                    and scale_now > 0.0
                    and float(lambda_norm) != 0.0
                    and constraint_loss_fn is not None
                ):
                    n_groups = next(iter(bank.values())).shape[0]
                    index = torch.randint(
                        n_groups,
                        (min(int(CLASS_TRAINING_CONFIG["constraint_batch_groups"]), n_groups),),
                        generator=constraint_generator,
                    )
                    mass_loss, _ = constraint_loss_fn(model, bank, index)
                bridge_loss = torch.zeros((), device=device)
                if (
                    bridge_bank is not None
                    and bridge_scale_now > 0.0
                    and float(lambda_bridge) != 0.0
                    and bridge_loss_fn is not None
                ):
                    n_groups = next(iter(bridge_bank.values())).shape[0]
                    index = torch.randint(
                        n_groups,
                        (min(int(CLASS_TRAINING_CONFIG["constraint_batch_groups"]), n_groups),),
                        generator=bridge_generator,
                    )
                    bridge_loss = bridge_loss_fn(model, bridge_bank, index)
                objective = (
                    ce
                    + direct_a_weight * (direct_a_bce - log_two)
                    + scale_now * float(lambda_norm) * mass_loss
                    + bridge_scale_now * float(lambda_bridge) * bridge_loss
                )
                if not torch.isfinite(objective):
                    raise FloatingPointError("Non-finite multiclass objective.")
                optimizer.zero_grad(set_to_none=True)
                objective.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLASS_TRAINING_CONFIG["gradient_clip"])
                optimizer.step()
                n_batch_groups = len(group_batch)
                n_batch_rows = n_batch_groups * int(n_classes)
                ce_sum += float(ce.detach().cpu()) * n_batch_rows
                direct_a_sum += float(direct_a_bce.detach().cpu()) * n_batch_groups
                mass_sum += float(mass_loss.detach().cpu()) * n_batch_groups
                bridge_sum += float(bridge_loss.detach().cpu()) * n_batch_groups
                total_sum += float(objective.detach().cpu()) * n_batch_groups
                n_rows += n_batch_rows
                n_groups_seen += n_batch_groups

            validation_ce, validation_a_bce = _validation_classifier_losses(
                model, validation_tensor, n_classes
            )
            if float(lambda_norm) != 0.0:
                validation_cross, validation_pooled = _validation_mass_metrics(
                    model, prepared_validation, constraint_loss_fn
                )
            else:
                validation_cross, validation_pooled = 0.0, 0.0
            if float(lambda_bridge) != 0.0:
                validation_bridge = _validation_bridge_metric(
                    model, prepared_bridge_validation, bridge_loss_fn
                )
            else:
                validation_bridge = 0.0
            validation_a_gain = log_two - validation_a_bce
            validation_total = (
                validation_ce
                + direct_a_weight * (validation_a_bce - log_two)
                + float(lambda_norm) * validation_cross
                + float(lambda_bridge) * validation_bridge
            )
            selection_active = bridge_scale_now >= 1.0

            history["train_ce"].append(ce_sum / max(1, n_rows))
            history["train_a_bce"].append(direct_a_sum / max(1, n_groups_seen))
            history["train_mass"].append(mass_sum / max(1, n_groups_seen))
            history["train_bridge"].append(bridge_sum / max(1, n_groups_seen))
            history["train_total"].append(total_sum / max(1, n_groups_seen))
            history["validation_ce"].append(validation_ce)
            history["validation_a_bce"].append(validation_a_bce)
            history["validation_a_gain"].append(validation_a_gain)
            history["validation_mass_cross"].append(validation_cross)
            history["validation_mass_pooled"].append(validation_pooled)
            history["validation_bridge"].append(validation_bridge)
            history["validation_total"].append(validation_total)
            history["constraint_scale"].append(scale_now)
            history["bridge_scale"].append(bridge_scale_now)
            history["learning_rate"].append(learning_rate)
            history["learning_rate_stage"].append(learning_rate_stage)

            minimum_improvement = float(CLASS_TRAINING_CONFIG["minimum_improvement"])
            if selection_active and validation_total < best_value - minimum_improvement:
                best_value = validation_total
                best_state = copy.deepcopy(model.state_dict())
                best_record = {
                    "epoch": epoch + 1,
                    "validation_ce": validation_ce,
                    "validation_a_bce": validation_a_bce,
                    "validation_a_gain": validation_a_gain,
                    "validation_mass_cross": validation_cross,
                    "validation_mass_pooled": validation_pooled,
                    "validation_bridge": validation_bridge,
                    "validation_total": validation_total,
                    "learning_rate": learning_rate,
                }
                stale = 0
            elif selection_active:
                stale += 1
            stage_changed = epoch == 0 or history["learning_rate_stage"][-1] != history["learning_rate_stage"][-2]
            if stage_changed or (epoch + 1) % max(1, CLASS_EPOCHS // 7) == 0:
                print(
                    f"member {member}, epoch {epoch + 1:3d}: CE={validation_ce:.6f}, "
                    f"a-BCE={validation_a_bce:.6f}, gain={validation_a_gain:+.3e}, "
                    f"C_Z={validation_cross:.4g}, C_B={validation_bridge:.4g}, "
                    f"lr={learning_rate:.1e}, ramps=({scale_now:.2f},{bridge_scale_now:.2f})"
                )
            final_lr_stage = len(CLASS_TRAINING_CONFIG["learning_rate_factors"]) - 1
            if (
                learning_rate_stage == final_lr_stage
                and stale >= int(CLASS_TRAINING_CONFIG["patience"])
                and selection_active
            ):
                break

        if best_state is None or best_record is None:
            raise RuntimeError("No finite classifier checkpoint was produced.")
        model.load_state_dict(best_state)
        model.eval()
        history["selected"] = [best_record]
        history["direct_a_ratio"] = direct_a_name
        torch.save({
            "state_dict": model.state_dict(),
            "center": center,
            "scale": robust_scale,
            "history": history,
            "n_classes": n_classes,
            "input_dim": class_points.shape[-1],
            "model_config": CLASS_MODEL_CONFIG,
            "fingerprint": fingerprint,
        }, checkpoint)
        ensemble.append({
            "model": model,
            "center": center,
            "scale": robust_scale,
            "history": history,
            "checkpoint": checkpoint,
        })
    return ensemble


@torch.no_grad()
def predict_shared_logits(ensemble, points, batch_size=65_536):
    points = _assert_finite("prediction points", points)
    original_shape = points.shape[:-1]
    flat = points.reshape(-1, points.shape[-1]).astype(np.float32)
    member_logits = []
    for pack in ensemble:
        transformed = _transform_classifier_points(flat, pack["center"], pack["scale"])
        chunks = []
        for start in range(0, len(flat), int(batch_size)):
            tensor = torch.as_tensor(
                transformed[start : start + int(batch_size)],
                dtype=torch.float32,
                device=device,
            )
            logits = pack["model"](tensor)
            logits = logits - logits.mean(dim=1, keepdim=True)
            chunks.append(logits.cpu().numpy())
        member_logits.append(np.concatenate(chunks, axis=0))
    averaged = np.mean(np.stack(member_logits), axis=0)
    return averaged.reshape(*original_shape, averaged.shape[-1])


@torch.no_grad()
def head_saturation_fraction(ensemble, points, head_name):
    points = _assert_finite("head diagnostic points", points)
    flat = points.reshape(-1, points.shape[-1]).astype(np.float32)
    saturated, count = 0, 0
    threshold = 0.8 * float(CLASS_MODEL_CONFIG["log_ratio_bound"])
    for pack in ensemble:
        transformed = _transform_classifier_points(flat, pack["center"], pack["scale"])
        for start in range(0, len(flat), 65_536):
            tensor = torch.as_tensor(
                transformed[start : start + 65_536],
                dtype=torch.float32,
                device=device,
            )
            raw = pack["model"].raw_ratio_heads(tensor)[head_name]
            saturated += int((raw.abs() > threshold).sum().cpu())
            count += int(raw.numel())
    return saturated / max(1, count)


def select_empirical_context_slices(context_pool):
    context_pool = _assert_finite("context pool", context_pool, ndim=2).astype(np.float32)
    center = np.median(context_pool, axis=0)
    mad = 1.4826 * np.median(np.abs(context_pool - center), axis=0)
    scale = np.where(mad > 1.0e-6, mad, context_pool.std(axis=0))
    scale = np.where(scale > 1.0e-6, scale, 1.0)
    score = np.max(np.abs((context_pool - center) / scale), axis=1)
    probabilities = np.array([0.50, 0.90, 0.97, 0.99, 0.995, 0.999, 0.9998, 1.0])
    targets = np.quantile(score, probabilities)
    indices = np.array([int(np.argmin(np.abs(score - value))) for value in targets])
    return context_pool[indices], score[indices], probabilities


def sample_only_flow_audit(flow_pack, contexts, truth_draws, scores, probabilities, name, seed):
    truth_draws = _assert_finite("simulator audit draws", truth_draws, ndim=3)
    flow_draws = _draw_conditional(flow_pack, contexts, truth_draws.shape[1], seed)
    rows = []
    quantiles = np.array([0.001, 0.01, 0.05, 0.50, 0.95, 0.99, 0.999])
    for index, (truth, learned) in enumerate(zip(truth_draws, flow_draws)):
        truth_scale = np.maximum(truth.std(axis=0), 1.0e-6)
        coordinate_w1 = np.array([
            wasserstein_distance(truth[:, feature], learned[:, feature])
            for feature in range(truth.shape[1])
        ]) / truth_scale
        mean_error = np.abs(learned.mean(axis=0) - truth.mean(axis=0)) / truth_scale
        quantile_error = np.max(np.abs(
            np.quantile(learned, quantiles, axis=0)
            - np.quantile(truth, quantiles, axis=0)
        ) / truth_scale, axis=0)

        projection_rng = np.random.default_rng(seed + 10_000 + index)
        directions = projection_rng.normal(size=(32, truth.shape[1]))
        directions /= np.linalg.norm(directions, axis=1, keepdims=True)
        truth_projection = truth @ directions.T
        learned_projection = learned @ directions.T
        projection_scale = np.maximum(truth_projection.std(axis=0), 1.0e-6)
        sliced_w1 = np.array([
            wasserstein_distance(truth_projection[:, direction], learned_projection[:, direction])
            for direction in range(len(directions))
        ]) / projection_scale

        truth_correlation = np.corrcoef(truth, rowvar=False)
        learned_correlation = np.corrcoef(learned, rowvar=False)
        correlation_error = np.max(np.abs(truth_correlation - learned_correlation))
        rows.append({
            "flow": name,
            "context quantile": float(probabilities[index]),
            "context score": float(scores[index]),
            "max coordinate W1 / truth sigma": float(np.max(coordinate_w1)),
            "q95 sliced W1 / truth sigma": float(np.quantile(sliced_w1, 0.95)),
            "max mean error / truth sigma": float(np.max(mean_error)),
            "max tail-quantile error / truth sigma": float(np.max(quantile_error)),
            "max correlation error": float(correlation_error),
        })
    result = pd.DataFrame(rows)
    if not np.isfinite(result.select_dtypes(include=[np.number])).all().all():
        raise FloatingPointError(f"{name}: non-finite sample-only flow audit.")
    print(
        f"{name} sample-only joint-tail audit: worst q95 sliced W1="
        f"{result['q95 sliced W1 / truth sigma'].max():.3f}."
    )
    return result


# Part I — nuisance marginalized implicitly

We first train two frozen **flow mixtures**,

$$q_P(\mu\mid x),\qquad q_L^m(x\mid\mu),$$

from independent simulator samples.  Their simulations contain $\alpha\sim\rho_\alpha$, but $\alpha$ is omitted from both targets and contexts, so this is genuinely nuisance-marginal learning.  The density used everywhere is the arithmetic mixture $M^{-1}\sum_mq_m$, and conditional draws select members uniformly.

The three balanced classifier classes are

$$
\pi_S=\rho_\mu(\mu)p_m(x\mid\mu),\quad
\pi_P=m_\rho(x)q_P(\mu\mid x),\quad
\pi_L=\rho_\mu(\mu)q_L^m(x\mid\mu).
$$

Consequently $e^{s_S-s_P}$ and $e^{s_S-s_L}$ are the posterior and likelihood residuals.  Every group contains one simulator row, one posterior-reference row at the same $x$, and one likelihood-reference row at the same $\mu$.

### Proposal-flow numerical contracts

The scalar $q_P$ and $q_N$ mixtures contain independently initialized four-layer monotone RQS members.  Each member undergoes exact inverse/forward preflight; no sample is clipped, rejected, or redrawn.  Scalar targets have no coordinates to mix.

The vector likelihood members build on Exercise 10's detailed flow with a full-support conditional affine entrance and a wider 80-bin, ten-coupling RQS stack.  Learned identity-initialized LU maps replace fixed permutations after every coupling.  We do **not** subtract the known Gaussian mean or use any density-aware residual.  All transforms and Jacobians are learned from samples.

After fitting, sample-only audits compare fresh flow-mixture and simulator draws at empirical context quantiles through coordinate-wise and sliced Wasserstein, mean, correlation, and tail-quantile errors.  Analytic likelihood values are not used.

### Direct posterior-ratio supervision

The $S$ and $P$ rows already present in each triplet form a leakage-safe binary task.  The same enlarged $a$-head is trained by both multinomial CE and a unit-weight balanced $S/P$ BCE.  This adds neither a density evaluation nor a simulated row; it makes the claimed finite-sample posterior ratio an explicit objective.


In [ ]:
# Independent simulations for the two frozen proposals.
rng = np.random.default_rng(SEED + 10)
theta_qp = sample_design(N_FLOW, rng)
x_qp = simulate(theta_qp, rng)
set_torch_seed(SEED + 11)
q_p = train_spline_flow_ensemble(
    theta_qp[:, :1],
    context=x_qp,
    checkpoint=MODEL_DIR / "q_p_mu_given_x_scalar_mixture_v4.pt",
    ensemble_size=FLOW_ENSEMBLE_SIZE,
    model_config=SCALAR_FLOW_MODEL_CONFIG,
    training_config=SCALAR_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 11,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qp, x_qp

rng = np.random.default_rng(SEED + 20)
theta_qlm = sample_vector_flow_design(N_FLOW, rng, score_columns=(0,))
# Only mu is a q_L^m context; alpha remains an ordinary design draw and
# is marginalized through simulation exactly as before.
theta_qlm[:, 1] = sample_alpha(N_FLOW, rng)
x_qlm = simulate(theta_qlm, rng)
set_torch_seed(SEED + 21)
q_lm = train_spline_flow_ensemble(
    x_qlm,
    context=theta_qlm[:, :1],
    checkpoint=MODEL_DIR / "q_lm_x_given_mu_full_support_lu_rqs_mixture_v4.pt",
    ensemble_size=FLOW_ENSEMBLE_SIZE,
    model_config=VECTOR_FLOW_MODEL_CONFIG,
    training_config=VECTOR_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 21,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qlm, x_qlm

# Sample-only q_L^m audit at central through extreme empirical mu ranks.
rng = np.random.default_rng(SEED + 22)
qlm_context_pool = sample_design(50_000 if not SMOKE_MODE else 2_000, rng)[:, :1]
qlm_audit_contexts, qlm_audit_scores, qlm_audit_probabilities = (
    select_empirical_context_slices(qlm_context_pool)
)
n_audit_contexts = len(qlm_audit_contexts)
mu_truth = np.repeat(qlm_audit_contexts[:, 0], N_FLOW_AUDIT_SAMPLES)
alpha_truth = sample_alpha(n_audit_contexts * N_FLOW_AUDIT_SAMPLES, rng)
qlm_truth_draws = simulate(
    np.column_stack([mu_truth, alpha_truth]), rng
).reshape(n_audit_contexts, N_FLOW_AUDIT_SAMPLES, 3)
qlm_tail_audit = sample_only_flow_audit(
    q_lm,
    qlm_audit_contexts,
    qlm_truth_draws,
    qlm_audit_scores,
    qlm_audit_probabilities,
    r"$q_L^m(x\mid\mu)$",
    SEED + 23,
)
display(qlm_tail_audit.style.format(precision=4))

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Preflight: invertibility before building classifier classes

This audit uses independent ordinary and extreme defensive-design contexts.  Every scalar-flow member must have strict quantile monotonicity, inverse/forward closure, finite log density, and exact repeated-seed sampling, with **zero** precision retries on this compact preflight.  The thresholds catch a stiff or collapsed inverse rather than grade density accuracy.  A failed float64 retry stops here; it is never converted into rejection sampling.  Mixture fidelity is assessed later by held-out closure, PIT, ESS, Pareto-$k$, and sample-only audits.


In [ ]:
rng = np.random.default_rng(SEED + 90)
theta_qp_audit = sample_design(16_384, rng)
theta_qp_tail = np.column_stack([
    np.array([-18, -15, -12, -10, 10, 12, 15, 18], dtype=np.float32),
    np.array([0, 6, -4, 3, -3, 4, -6, 0], dtype=np.float32),
])
x_qp_audit = np.concatenate([
    simulate(theta_qp_audit, rng),
    simulate(theta_qp_tail, rng),
])
for member_index, member in enumerate(q_p):
    audit_scalar_flow(
        member, x_qp_audit, rf"$q_P^{{({member_index})}}(\mu\mid x)$",
        SEED + 91 + 100 * member_index,
    )
del theta_qp_audit, theta_qp_tail, x_qp_audit


In [ ]:
def build_three_class_groups(n_groups, q_p, q_lm, seed):
    rng = np.random.default_rng(seed)
    theta_s = sample_design(n_groups, rng)
    mu_s = theta_s[:, :1]
    x_s = simulate(theta_s, rng)
    mu_p = _draw_conditional(q_p, x_s, 1, seed + 1)[:, 0, :]
    x_l = _draw_conditional(q_lm, mu_s, 1, seed + 2)[:, 0, :]
    points_s = np.column_stack([mu_s, x_s])
    points_p = np.column_stack([mu_p, x_s])
    points_l = np.column_stack([mu_s, x_l])
    groups = np.stack([points_s, points_p, points_l], axis=1).astype(np.float32)
    return _assert_finite("three-class groups", groups, ndim=3)

three_retry_before = _rqs_retry_count()
three_class_groups = build_three_class_groups(
    N_CLASS, q_p, q_lm, SEED + 100
)
print("Three-class grouped tensor:", three_class_groups.shape)
print(
    "Inverse-RQS kernel calls retried in float64 during three-class construction:",
    _rqs_retry_count() - three_retry_before,
)


## Cross-fitted conditional normalization and the bridge

For $(S,P,L)$ the conditional masses are

$$Z_P(x)=\mathbb E_{q_P(\mu\mid x)}e^{s_S-s_P},\qquad
Z_L(\mu)=\mathbb E_{q_L^m(x\mid\mu)}e^{s_S-s_L}.$$

Independent A/B proposal draws give an unbiased cross-mass estimate of $(Z-1)^2$.  The proposal in every formula is the actual arithmetic flow mixture, $q_{\rm mix}=M^{-1}\sum_mq_m$; averaging member log densities would describe a different distribution.

The raw three-class bridge is

$$
\ell_B(\mu,x)=\log\rho_\mu(\mu)+\log q_L^m(x\mid\mu)
-\log q_P(\mu\mid x)+s_P-s_L.
$$

At the population solution it equals $\log m_\rho(x)$ and therefore cannot depend on $\mu$ at fixed $x$.  We minimize the ordinary unbiased sample variance over **IID** arithmetic-mixture proposal draws.  Balanced member allocation remains useful for normalization moments but is not used inside this variance.  Proposal densities are fixed loss metadata, never classifier inputs.  No simulator density or nested $\log\widehat Z$ appears in this training term.


In [ ]:
def build_three_normalization_bundle(n_groups, n_inner, q_p, q_lm, seed):
    rng = np.random.default_rng(seed)

    theta_x = sample_tail_enriched_design(n_groups, rng)
    x_zp = simulate(theta_x, rng)
    mu_zp_a = _draw_conditional(q_p, x_zp, n_inner, seed + 1)
    mu_zp_b = _draw_conditional(q_p, x_zp, n_inner, seed + 2)
    x_zp_repeat = np.repeat(x_zp[:, None, :], n_inner, axis=1)

    mu_zl = sample_tail_enriched_design(n_groups, rng)[:, :1]
    x_zl_a = _draw_conditional(q_lm, mu_zl, n_inner, seed + 3)
    x_zl_b = _draw_conditional(q_lm, mu_zl, n_inner, seed + 4)
    mu_zl_repeat = np.repeat(mu_zl[:, None, :], n_inner, axis=1)

    bundle = {
        "zp_a_points": np.concatenate([mu_zp_a, x_zp_repeat], axis=2),
        "zp_b_points": np.concatenate([mu_zp_b, x_zp_repeat], axis=2),
        "zl_a_points": np.concatenate([mu_zl_repeat, x_zl_a], axis=2),
        "zl_b_points": np.concatenate([mu_zl_repeat, x_zl_b], axis=2),
    }
    for name, value in bundle.items():
        _assert_finite(name, value)
        if len(value) != n_groups:
            raise RuntimeError(f"{name} lost its anchor-group axis.")
    return bundle


def three_normalization_loss(model, bundle, index):
    logits_zp_a = _constraint_logits(model, bundle, "zp_a_points", index, 3)
    logits_zp_b = _constraint_logits(model, bundle, "zp_b_points", index, 3)
    cross_zp, monitor_zp = _partition_cross_mass(logits_zp_a, logits_zp_b, 0, 1)

    logits_zl_a = _constraint_logits(model, bundle, "zl_a_points", index, 3)
    logits_zl_b = _constraint_logits(model, bundle, "zl_b_points", index, 3)
    cross_zl, monitor_zl = _partition_cross_mass(logits_zl_a, logits_zl_b, 0, 2)
    return 0.5 * (cross_zp + cross_zl), 0.5 * (monitor_zp + monitor_zl)


# Train/validation normalization and bridge banks occupy disjoint seed
# namespaces; the stride leaves room for every builder's internal offsets.
constraints_three_train = [
    build_three_normalization_bundle(
        N_NORM_TRAIN_GROUPS,
        N_NORM_TRAIN_INNER,
        q_p,
        q_lm,
        SEED + 20_000 + 100 * bank,
    )
    for bank in range(N_NORM_TRAIN_BANKS)
]
constraints_three_validation = [
    build_three_normalization_bundle(
        N_NORM_VALID_GROUPS,
        N_NORM_VALID_INNER,
        q_p,
        q_lm,
        SEED + 30_000 + 100 * bank,
    )
    for bank in range(N_NORM_VALID_BANKS)
]


def build_three_bridge_bundle(n_groups, n_inner, q_p, q_lm, seed):
    if int(n_inner) < 2:
        raise ValueError("Bridge variance requires at least two draws per anchor.")
    rng = np.random.default_rng(seed)
    theta_anchor = sample_tail_enriched_design(n_groups, rng)
    x_anchor = simulate(theta_anchor, rng)
    mu = _draw_conditional(
        q_p, x_anchor, n_inner, seed + 1, allocation="iid"
    )
    x_repeat = np.repeat(x_anchor[:, None, :], n_inner, axis=1)
    points = np.concatenate([mu, x_repeat], axis=2).astype(np.float32)
    flat_mu = mu.reshape(-1, 1)
    flat_x = x_repeat.reshape(-1, 3)
    bridge_base = (
        design_mu_logpdf(flat_mu[:, 0])
        + _flow_log_prob(q_lm, flat_x, context=flat_mu)
        - _flow_log_prob(q_p, flat_mu, context=flat_x)
    ).reshape(n_groups, n_inner).astype(np.float32)
    return {
        "bridge_points": _assert_finite("three bridge points", points),
        "bridge_base": _assert_finite("three bridge base", bridge_base),
    }


def three_bridge_loss(model, bundle, index):
    logits = _constraint_logits(model, bundle, "bridge_points", index, 3)
    base = bundle["bridge_base"].index_select(0, index).to(device)
    log_evidence = base + logits[..., 1] - logits[..., 2]
    return log_evidence.var(dim=1, unbiased=True).mean()


bridge_three_train = [
    build_three_bridge_bundle(
        N_BRIDGE_TRAIN_GROUPS,
        N_BRIDGE_TRAIN_INNER,
        q_p,
        q_lm,
        SEED + 40_000 + 100 * bank,
    )
    for bank in range(N_NORM_TRAIN_BANKS)
]
bridge_three_validation = [
    build_three_bridge_bundle(
        N_BRIDGE_VALID_GROUPS,
        N_BRIDGE_VALID_INNER,
        q_p,
        q_lm,
        SEED + 50_000 + 100 * bank,
    )
    for bank in range(N_NORM_VALID_BANKS)
]


## Controlled three-objective ablation

All three ensembles below use the same proposal mixtures, sampled triplets, group split, model initialization seeds and learning-rate schedule, direct $S/P$ proper score, and frozen normalization/bridge banks.  “CE” is the common multiclass-plus-direct-$a$ objective.  Only the two displayed coefficients change (their optimizer states naturally diverge once the objectives diverge):

| label | $\lambda_Z$ | $\lambda_B$ |
|---|---:|---:|
| CE | 0 | 0 |
| CE–normalization | 0.20 | 0 |
| CE–normalization–bridge | 0.20 | 0.05 |

This is a training comparison, not a proof that either regularizer must improve simulator fidelity.


In [ ]:
three_direct = train_multiclass_ensemble(
    three_class_groups,
    n_classes=3,
    checkpoint_dir=MODEL_DIR / "three_class_ce",
    seed_base=SEED + 400,
    constraint_train=constraints_three_train,
    constraint_validation=constraints_three_validation,
    constraint_loss_fn=three_normalization_loss,
    lambda_norm=0.0,
    bridge_train=bridge_three_train,
    bridge_validation=bridge_three_validation,
    bridge_loss_fn=three_bridge_loss,
    lambda_bridge=0.0,
)

three_normalized = train_multiclass_ensemble(
    three_class_groups,
    n_classes=3,
    checkpoint_dir=MODEL_DIR / "three_class_ce_normalization",
    seed_base=SEED + 400,
    constraint_train=constraints_three_train,
    constraint_validation=constraints_three_validation,
    constraint_loss_fn=three_normalization_loss,
    lambda_norm=LAMBDA_NORM_3,
    bridge_train=bridge_three_train,
    bridge_validation=bridge_three_validation,
    bridge_loss_fn=three_bridge_loss,
    lambda_bridge=0.0,
)

three_bridge = train_multiclass_ensemble(
    three_class_groups,
    n_classes=3,
    checkpoint_dir=MODEL_DIR / "three_class_ce_normalization_bridge",
    seed_base=SEED + 400,
    constraint_train=constraints_three_train,
    constraint_validation=constraints_three_validation,
    constraint_loss_fn=three_normalization_loss,
    lambda_norm=LAMBDA_NORM_3,
    bridge_train=bridge_three_train,
    bridge_validation=bridge_three_validation,
    bridge_loss_fn=three_bridge_loss,
    lambda_bridge=LAMBDA_BRIDGE_3,
)


## Independent closure of the three-class experiment

The next cells evaluate all three objectives with fresh proposal draws and analytic truth that never entered training.  We compare posterior and likelihood routes, fully normalized implied evidence, conditional masses, PIT/coverage, importance tails, and saturation.  The bridge curve includes fresh $\log Z_P-\log Z_L$ corrections; it is not the raw training statistic.


In [ ]:
def three_log_zp(ensemble, x_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    mu = _draw_conditional(q_p, x_values, n_reference, seed)
    points = np.concatenate(
        [mu, np.repeat(x_values[:, None, :], n_reference, axis=1)], axis=2
    )
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 0] - logits[..., 1], axis=1) - np.log(n_reference)

def three_log_zl(ensemble, mu_values, n_reference, seed):
    mu_values = np.asarray(mu_values, dtype=np.float32).reshape(-1, 1)
    x = _draw_conditional(q_lm, mu_values, n_reference, seed)
    points = np.concatenate(
        [np.repeat(mu_values[:, None, :], n_reference, axis=1), x], axis=2
    )
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 0] - logits[..., 2], axis=1) - np.log(n_reference)

def three_closure(ensemble, mu_grid, x_observed, n_reference, seed):
    mu_grid = np.asarray(mu_grid, dtype=float)
    x_grid = np.repeat(np.asarray(x_observed)[None, :], len(mu_grid), axis=0)
    points = np.column_stack([mu_grid, x_grid]).astype(np.float32)
    logits = predict_shared_logits(ensemble, points)
    log_qp = _flow_log_prob(q_p, mu_grid[:, None], context=x_grid)
    log_ql = _flow_log_prob(q_lm, x_grid, context=mu_grid[:, None])
    log_zp = float(three_log_zp(ensemble, x_observed, n_reference, seed)[0])
    log_zl = three_log_zl(ensemble, mu_grid, n_reference, seed + 1)

    log_posterior_route = log_qp + logits[:, 0] - logits[:, 1] - log_zp
    posterior_route, _ = normalize_log_curve(log_posterior_route, mu_grid)
    log_likelihood_route = log_ql + logits[:, 0] - logits[:, 2] - log_zl
    likelihood_posterior, _ = normalize_log_curve(
        design_mu_logpdf(mu_grid) + log_likelihood_route, mu_grid
    )
    log_evidence_consistency = (
        design_mu_logpdf(mu_grid)
        + log_ql
        - log_qp
        + logits[:, 1]
        - logits[:, 2]
        + log_zp
        - log_zl
    )
    return {
        "posterior": posterior_route,
        "likelihood_posterior": likelihood_posterior,
        "log_evidence_consistency": log_evidence_consistency,
        "log_zp": log_zp,
        "log_zl": log_zl,
        "log_ratio_grid": logits[:, 0] - logits[:, 1],
    }

MU_GRID = np.linspace(-3.8, 3.8, 321)
truth_mu, _ = normalize_log_curve(
    design_mu_logpdf(MU_GRID) + marginal_log_likelihood(X_OBS, MU_GRID),
    MU_GRID,
)
MU_EVIDENCE_GRID = np.linspace(-10.0, 10.0, 4001)
_, LOG_EVIDENCE_TRUTH = normalize_log_curve(
    design_mu_logpdf(MU_EVIDENCE_GRID)
    + marginal_log_likelihood(X_OBS, MU_EVIDENCE_GRID),
    MU_EVIDENCE_GRID,
)
q_p_curve, _ = normalize_log_curve(
    _flow_log_prob(
        q_p,
        MU_GRID[:, None],
        context=np.repeat(X_OBS[None, :], len(MU_GRID), axis=0),
    ),
    MU_GRID,
)
closure_direct = three_closure(
    three_direct, MU_GRID, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 500
)
closure_normalized = three_closure(
    three_normalized, MU_GRID, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 500
)
closure_bridge = three_closure(
    three_bridge, MU_GRID, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 500
)

# Held-out conditional normalizers at many independent contexts.
rng = np.random.default_rng(SEED + 510)
n_context_check = 36 if SMOKE_MODE else (80 if FAST_MODE else 160)
theta_check = sample_design(n_context_check, rng)
x_check = simulate(theta_check, rng)
mu_check = sample_mu(n_context_check, rng)
heldout_norm = {}
for name, ensemble in [(THREE_DIRECT_LABEL, three_direct), (THREE_NORMALIZED_LABEL, three_normalized), (THREE_BRIDGE_LABEL, three_bridge)]:
    heldout_norm[name] = {
        "log_zp": three_log_zp(
            ensemble, x_check, N_DIAGNOSTIC_REFERENCE, SEED + 520
        ),
        "log_zl": three_log_zl(
            ensemble, mu_check, N_DIAGNOSTIC_REFERENCE, SEED + 521
        ),
    }

# Importance-tail diagnostics at x_obs.
mu_tail = _draw_conditional(
    q_p, X_OBS[None, :],
    2_000 if SMOKE_MODE else (20_000 if FAST_MODE else 80_000),
    SEED + 530,
)[0]
x_tail = np.repeat(X_OBS[None, :], len(mu_tail), axis=0)
tail_points = np.column_stack([mu_tail, x_tail])
tail_summaries = {}
tail_log_weights = {}
for name, ensemble in [(THREE_DIRECT_LABEL, three_direct), (THREE_NORMALIZED_LABEL, three_normalized), (THREE_BRIDGE_LABEL, three_bridge)]:
    logits = predict_shared_logits(ensemble, tail_points)
    log_weights = logits[:, 0] - logits[:, 1]
    tail_log_weights[name] = log_weights
    tail_summaries[name] = importance_tail_summary(log_weights)

mu_likelihood_tail = float(MU_GRID[np.argmax(truth_mu)])
x_likelihood_tail = _draw_conditional(
    q_lm,
    [[mu_likelihood_tail]],
    len(mu_tail),
    SEED + 531,
)[0]
likelihood_tail_points = np.column_stack([
    np.full(len(x_likelihood_tail), mu_likelihood_tail),
    x_likelihood_tail,
])
likelihood_tail_summaries = {}
for name, ensemble in [(THREE_DIRECT_LABEL, three_direct), (THREE_NORMALIZED_LABEL, three_normalized), (THREE_BRIDGE_LABEL, three_bridge)]:
    logits = predict_shared_logits(ensemble, likelihood_tail_points)
    likelihood_tail_summaries[name] = importance_tail_summary(
        logits[:, 0] - logits[:, 2]
    )

def heldout_three_bridge_rms(ensemble, banks):
    values = []
    for bundle in banks:
        logits = predict_shared_logits(ensemble, bundle["bridge_points"])
        implied = bundle["bridge_base"] + logits[..., 1] - logits[..., 2]
        values.append(np.std(implied, axis=1, ddof=1))
    return np.concatenate(values)


heldout_bridge_rms = {
    name: heldout_three_bridge_rms(ensemble, bridge_three_validation)
    for name, ensemble in [
        (THREE_DIRECT_LABEL, three_direct),
        (THREE_NORMALIZED_LABEL, three_normalized),
        (THREE_BRIDGE_LABEL, three_bridge),
    ]
}


rows = []
for name, closure, ensemble in [
    (THREE_DIRECT_LABEL, closure_direct, three_direct),
    (THREE_NORMALIZED_LABEL, closure_normalized, three_normalized),
    (THREE_BRIDGE_LABEL, closure_bridge, three_bridge),
]:
    normalizers = heldout_norm[name]
    support_mask = truth_mu > 1.0e-4 * truth_mu.max()
    consistency_delta = closure["log_evidence_consistency"] - LOG_EVIDENCE_TRUTH
    tails = tail_summaries[name]
    all_log_z = np.concatenate([normalizers["log_zp"], normalizers["log_zl"]])
    rows.append({
        "objective": name,
        "posterior IAE": integrated_absolute_error(truth_mu, closure["posterior"], MU_GRID),
        "likelihood-route IAE": integrated_absolute_error(
            truth_mu, closure["likelihood_posterior"], MU_GRID
        ),
        "evidence RMS (supported)": float(np.sqrt(np.mean(consistency_delta[support_mask] ** 2))),
        "bridge RMS (centered path)": float(np.std(
            closure["log_evidence_consistency"][support_mask], ddof=1
        )),
        "held-out raw bridge median RMS": float(np.median(heldout_bridge_rms[name])),
        "held-out raw bridge q95 RMS": float(np.quantile(heldout_bridge_rms[name], 0.95)),
        "RMS log Z": float(np.sqrt(np.mean(all_log_z**2))),
        "q95 |log Z|": float(np.quantile(np.abs(all_log_z), 0.95)),
        "max |log Z|": float(np.max(np.abs(all_log_z))),
        "a-head saturation (q_P)": head_saturation_fraction(
            ensemble, tail_points, "a"
        ),
        "c-head saturation (q_Lm)": head_saturation_fraction(
            ensemble, likelihood_tail_points, "c"
        ),
        "ESS fraction": tails["ESS_fraction"],
        "Pareto k": tails["pareto_k"],
        "max weight": tails["max_weight_fraction"],
        "likelihood ESS fraction": likelihood_tail_summaries[name]["ESS_fraction"],
        "likelihood Pareto k": likelihood_tail_summaries[name]["pareto_k"],
        "likelihood max weight": likelihood_tail_summaries[name]["max_weight_fraction"],
    })
three_summary = pd.DataFrame(rows)
display(three_summary.style.format(precision=4))


In [ ]:
DIRECT_COLOR = "#D55E00"
NORM_COLOR = "#0072B2"
FULL3_COLOR = "#009E73"
FULL4_COLOR = "#6A3D9A"

three_plot_models = [
    (THREE_DIRECT_LABEL, closure_direct, three_direct, DIRECT_COLOR),
    (THREE_NORMALIZED_LABEL, closure_normalized, three_normalized, NORM_COLOR),
    (THREE_BRIDGE_LABEL, closure_bridge, three_bridge, FULL3_COLOR),
]

fig, axes = plt.subplots(2, 2, figsize=(12.4, 8.9), constrained_layout=True)
axes[0, 0].plot(MU_GRID, truth_mu, color="black", lw=2.4, label="analytic truth")
axes[0, 0].plot(MU_GRID, q_p_curve, color="0.6", lw=1.5, ls=":", label=r"proposal $q_P$")
for name, closure, _, color in three_plot_models:
    axes[0, 0].plot(MU_GRID, closure["posterior"], color=color, lw=2.0, label=name)
axes[0, 0].plot(
    MU_GRID, closure_bridge["likelihood_posterior"], color=FULL3_COLOR,
    lw=1.8, ls="--", label="full objective: likelihood route",
)
axes[0, 0].set(xlabel=r"$\mu$", ylabel="posterior density", title="(a) Posterior closure")
axes[0, 0].legend(fontsize=8)

supported = truth_mu > 1.0e-4 * truth_mu.max()
for name, closure, _, color in three_plot_models:
    residual = closure["log_evidence_consistency"] - LOG_EVIDENCE_TRUTH
    axes[0, 1].plot(MU_GRID, np.where(supported, residual, np.nan), color=color, lw=2, label=name)
axes[0, 1].axhline(0.0, color="black", lw=1)
axes[0, 1].set(
    xlabel=r"$\mu$", ylabel=r"$\log\widehat m-\log m_{\rm true}$",
    title="(b) Fresh normalized evidence bridge",
)
axes[0, 1].legend(fontsize=8)

jitter = {THREE_DIRECT_LABEL: -0.09, THREE_NORMALIZED_LABEL: 0.0, THREE_BRIDGE_LABEL: 0.09}
color_map = {THREE_DIRECT_LABEL: DIRECT_COLOR, THREE_NORMALIZED_LABEL: NORM_COLOR, THREE_BRIDGE_LABEL: FULL3_COLOR}
for model_index, (name, _, _, _) in enumerate(three_plot_models):
    values_p = heldout_norm[name]["log_zp"]
    values_l = heldout_norm[name]["log_zl"]
    rng_plot = np.random.default_rng(SEED + 1 + model_index)
    axes[1, 0].scatter(
        np.full_like(values_p, jitter[name]) + rng_plot.normal(0, 0.012, len(values_p)),
        values_p, s=12, alpha=0.42, color=color_map[name],
    )
    axes[1, 0].scatter(
        np.full_like(values_l, 1.0 + jitter[name]) + rng_plot.normal(0, 0.012, len(values_l)),
        values_l, s=12, alpha=0.42, color=color_map[name], label=name,
    )
axes[1, 0].axhline(0.0, color="black", lw=1)
axes[1, 0].set(
    xticks=[0, 1], xticklabels=[r"$\log Z_P(x)$", r"$\log Z_L(\mu)$"],
    ylabel="held-out conditional log normalizer", title="(c) Independent mass closure",
)
axes[1, 0].legend(fontsize=8)

for name, _, _, color in three_plot_models:
    log_w = tail_log_weights[name]
    weights = np.exp(log_w - logsumexp(log_w)) * len(log_w)
    ordered = np.sort(np.maximum(weights, np.finfo(float).tiny))
    survival = 1.0 - np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
    axes[1, 1].plot(ordered, survival, color=color, lw=1.8, label=name)
axes[1, 1].set(
    xscale="log", yscale="log", xlabel=r"normalized correction $r_P/\bar r_P$",
    ylabel="empirical survival", title="(d) Posterior-correction tail",
)
axes[1, 1].legend(fontsize=8)
for ax in axes.flat:
    ax.grid(alpha=0.25)
export_exercise9_multiclass_figure(fig, "three_class_ce_vs_normalization_vs_bridge")
plt.show()


**Figure 1 interpretation.**  The curves isolate the two consistency regularizers on top of a common multiclass-plus-direct-$a$ objective.  $\mathcal L_Z$ targets conditional mass; $\mathcal L_B$ targets parameter-invariance of the jointly implied evidence.  Neither guarantees agreement with the analytic simulator.  The supported-region evidence residual, fresh $Z$ banks, PIT/coverage, and importance-tail panels determine whether a smaller training bridge corresponds to useful closure.


## Held-out simulation-based calibration of all three objectives

Closure at one $x_{\rm obs}$ is not evidence of amortized calibration.  We draw a new design sample $(\mu_i,\alpha_i,x_i)$ and compute posterior PIT values from the proposal and all three residual classifiers.  A calibrated design posterior has uniform PIT values and nominal equal-tailed coverage.

Full mode uses 2,000 contexts.  A paper result must repeat the complete flow/classifier training over independent seeds and show paired uncertainty bands.


In [ ]:
rng = np.random.default_rng(SEED + 1300)
theta_calibration = sample_design(N_CALIBRATION_CONTEXTS, rng)
x_calibration = simulate(theta_calibration, rng)
mu_calibration = _draw_conditional(
    q_p, x_calibration, N_CALIBRATION_SAMPLES, SEED + 1301
)[..., 0]
calibration_points = np.concatenate([
    mu_calibration[..., None],
    np.repeat(x_calibration[:, None, :], N_CALIBRATION_SAMPLES, axis=1),
], axis=2)
below_truth = mu_calibration <= theta_calibration[:, 0, None]

pit_values = {"proposal q_P": below_truth.mean(axis=1)}
for name, ensemble in [
    (THREE_DIRECT_LABEL, three_direct),
    (THREE_NORMALIZED_LABEL, three_normalized),
    (THREE_BRIDGE_LABEL, three_bridge),
]:
    logits = predict_shared_logits(ensemble, calibration_points)
    log_weights = logits[..., 0] - logits[..., 1]
    weights = np.exp(log_weights - logsumexp(log_weights, axis=1, keepdims=True))
    pit_values[name] = np.sum(weights * below_truth, axis=1)

NOMINAL_COVERAGE = np.linspace(0.05, 0.95, 19)
coverage_values = {
    name: np.array([
        np.mean((pit >= 0.5 * (1.0 - level)) & (pit <= 0.5 * (1.0 + level)))
        for level in NOMINAL_COVERAGE
    ])
    for name, pit in pit_values.items()
}
calibration_rows = []
for name, pit in pit_values.items():
    ordered = np.sort(pit)
    uniform_quantiles = (np.arange(len(ordered)) + 0.5) / len(ordered)
    calibration_rows.append({
        "method": name,
        "PIT KS distance": float(np.max(np.abs(ordered - uniform_quantiles))),
        "max coverage error": float(np.max(np.abs(coverage_values[name] - NOMINAL_COVERAGE))),
    })
calibration_summary = pd.DataFrame(calibration_rows)
display(calibration_summary.style.format(precision=4))

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.3), constrained_layout=True)
calibration_colors = {
    "proposal q_P": "0.55",
    THREE_DIRECT_LABEL: DIRECT_COLOR,
    THREE_NORMALIZED_LABEL: NORM_COLOR,
    THREE_BRIDGE_LABEL: FULL3_COLOR,
}
for name, pit in pit_values.items():
    ordered = np.sort(pit)
    empirical_cdf = np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
    axes[0].plot(ordered, empirical_cdf, lw=1.9, color=calibration_colors[name], label=name)
    axes[1].plot(
        NOMINAL_COVERAGE, coverage_values[name], lw=1.9,
        color=calibration_colors[name], label=name,
    )
axes[0].plot([0, 1], [0, 1], color="black", ls="--", lw=1)
axes[0].set(
    xlabel="posterior PIT", ylabel="empirical CDF",
    title="(a) Held-out posterior PIT",
)
axes[1].plot([0, 1], [0, 1], color="black", ls="--", lw=1)
binomial_sigma = np.sqrt(
    NOMINAL_COVERAGE * (1.0 - NOMINAL_COVERAGE) / N_CALIBRATION_CONTEXTS
)
axes[1].fill_between(
    NOMINAL_COVERAGE,
    np.maximum(0.0, NOMINAL_COVERAGE - binomial_sigma),
    np.minimum(1.0, NOMINAL_COVERAGE + binomial_sigma),
    color="0.7", alpha=0.2, linewidth=0, label=r"$\pm1\sigma$ binomial",
)
axes[1].set(
    xlabel="nominal equal-tailed coverage", ylabel="empirical coverage",
    title="(b) Held-out coverage",
)
for ax in axes:
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
export_exercise9_multiclass_figure(fig, "three_class_heldout_calibration")
plt.show()


# Part II — the nuisance parameter is explicit

We now train

$$q_N(\alpha\mid x,\mu),\qquad q_L(x\mid\mu,\alpha),$$

while reusing the frozen marginal $q_P(\mu\mid x)$.  The four balanced classes are

$$
\begin{aligned}
\Pi_S &= \rho_\mu\rho_\alpha p(x\mid\mu,\alpha),\\
\Pi_N &= \rho_\mu p_m(x\mid\mu)q_N(\alpha\mid x,\mu),\\
\Pi_P &= m_\rho(x)q_P(\mu\mid x)q_N(\alpha\mid x,\mu),\\
\Pi_L &= \rho_\mu\rho_\alpha q_L(x\mid\mu,\alpha).
\end{aligned}
$$

The ratio-head architecture supplies $r_P=e^a$, $r_N=e^b$, $r_L=e^c$, and $r_{PN}=e^{a+b}$.  Only $a$ excludes $\alpha$: this encodes the exact cancellation of the shared $q_N$ factor between classes $N$ and $P$.  The network is not given $p$, $q$, $\rho$, an analytic mean, or any log density as an input.

Because $a=\log(N/P)$ in this factorization, its head-local binary task uses $N$ as the positive class and $P$ as the negative class.  Calling this an $S/P$ loss would be incorrect: the literal joint posterior ratio is $a+b$.  Directly supervising $N/P$ is precisely what targets the marginal-$\mu$ correction while preserving the architectural cancellation of $q_N$.


In [ ]:
# q_L^m is no longer needed for nuisance-aware training.  Keep its pack
# usable for rerunning Part I, but release GPU memory.
for member in q_lm:
    member["flow"].to(torch.device("cpu"))
if torch.cuda.is_available():
    torch.cuda.empty_cache()

rng = np.random.default_rng(SEED + 600)
theta_qn = sample_design(N_FLOW, rng)
x_qn = simulate(theta_qn, rng)
set_torch_seed(SEED + 601)
q_n = train_spline_flow_ensemble(
    theta_qn[:, 1:2],
    context=np.column_stack([x_qn, theta_qn[:, 0]]),
    checkpoint=MODEL_DIR / "q_n_alpha_given_x_mu_scalar_mixture_v4.pt",
    ensemble_size=FLOW_ENSEMBLE_SIZE,
    model_config=SCALAR_FLOW_MODEL_CONFIG,
    training_config=SCALAR_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 601,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qn, x_qn

rng = np.random.default_rng(SEED + 610)
theta_ql = sample_vector_flow_design(N_FLOW, rng, score_columns=(0, 1))
x_ql = simulate(theta_ql, rng)
set_torch_seed(SEED + 611)
q_l = train_spline_flow_ensemble(
    x_ql,
    context=theta_ql,
    checkpoint=MODEL_DIR / "q_l_x_given_mu_alpha_full_support_lu_rqs_mixture_v4.pt",
    ensemble_size=FLOW_ENSEMBLE_SIZE,
    model_config=VECTOR_FLOW_MODEL_CONFIG,
    training_config=VECTOR_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 611,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_ql, x_ql

# The same sample-only tail audit, now at explicit (mu,alpha) contexts.
rng = np.random.default_rng(SEED + 612)
ql_context_pool = sample_design(50_000 if not SMOKE_MODE else 2_000, rng)
ql_audit_contexts, ql_audit_scores, ql_audit_probabilities = (
    select_empirical_context_slices(ql_context_pool)
)
n_audit_contexts = len(ql_audit_contexts)
ql_truth_draws = simulate(
    np.repeat(ql_audit_contexts, N_FLOW_AUDIT_SAMPLES, axis=0), rng
).reshape(n_audit_contexts, N_FLOW_AUDIT_SAMPLES, 3)
ql_tail_audit = sample_only_flow_audit(
    q_l,
    ql_audit_contexts,
    ql_truth_draws,
    ql_audit_scores,
    ql_audit_probabilities,
    r"$q_L(x\mid\mu,\alpha)$",
    SEED + 613,
)
display(ql_tail_audit.style.format(precision=4))

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.3), constrained_layout=True)
for table, color, label in [
    (qlm_tail_audit, FULL3_COLOR, r"$q_L^m$"),
    (ql_tail_audit, FULL4_COLOR, r"$q_L$"),
]:
    axes[0].plot(
        table["context score"], table["q95 sliced W1 / truth sigma"],
        marker="o", lw=1.8, color=color, label=label,
    )
    axes[1].plot(
        table["context score"], table["max tail-quantile error / truth sigma"],
        marker="o", lw=1.8, color=color, label=label,
    )
axes[0].set(
    xlabel="empirical context-tail score", ylabel="q95 sliced W1 / truth sigma",
    title="(a) Joint sample-only transport",
)
axes[1].set(
    xlabel="empirical context-tail score", ylabel="max tail-quantile error / truth sigma",
    title="(b) 0.1% through 99.9% quantiles",
)
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8.5)
export_exercise9_multiclass_figure(fig, "vector_flow_sample_only_tail_audit")
plt.show()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Preflight for the explicit nuisance proposal

The same memberwise scalar-flow contract is applied to every $q_N(\alpha\mid x,\mu)$ mixture component before any four-class rows or nested posterior references are generated.


In [ ]:
rng = np.random.default_rng(SEED + 618)
theta_qn_audit = sample_design(16_384, rng)
theta_qn_tail = np.column_stack([
    np.array([-10, -8, -5, 0, 0, 5, 8, 10], dtype=np.float32),
    np.array([-14, 12, -10, -8, 8, 10, -12, 14], dtype=np.float32),
])
theta_qn_all = np.concatenate([theta_qn_audit, theta_qn_tail])
x_qn_audit = simulate(theta_qn_all, rng)
qn_audit_context = np.column_stack([x_qn_audit, theta_qn_all[:, 0]])
for member_index, member in enumerate(q_n):
    audit_scalar_flow(
        member,
        qn_audit_context,
        rf"$q_N^{{({member_index})}}(\alpha\mid x,\mu)$",
        SEED + 619 + 100 * member_index,
    )
del theta_qn_audit, theta_qn_tail, theta_qn_all, x_qn_audit, qn_audit_context


In [ ]:
def _qn_context(x, mu):
    x = np.atleast_2d(np.asarray(x, dtype=np.float32))
    mu = np.asarray(mu, dtype=np.float32).reshape(-1, 1)
    if len(x) == 1 and len(mu) > 1:
        x = np.repeat(x, len(mu), axis=0)
    if len(x) != len(mu):
        raise ValueError("x and mu rows must match.")
    return np.column_stack([x, mu])

def _draw_joint_reference(x, n_samples, seed):
    x = np.atleast_2d(np.asarray(x, dtype=np.float32))
    mu = _draw_conditional(q_p, x, n_samples, seed)
    x_repeat = np.repeat(x[:, None, :], n_samples, axis=1)
    flat_context = _qn_context(x_repeat.reshape(-1, 3), mu.reshape(-1, 1))
    alpha = _draw_conditional(q_n, flat_context, 1, seed + 1)[:, 0, :]
    alpha = alpha.reshape(len(x), n_samples, 1)
    return mu, alpha

def build_four_class_groups(n_groups, q_p, q_n, q_l, seed):
    rng = np.random.default_rng(seed)
    theta_s = sample_design(n_groups, rng)
    mu_s, alpha_s = theta_s[:, :1], theta_s[:, 1:2]
    x_s = simulate(theta_s, rng)

    alpha_n = _draw_conditional(q_n, _qn_context(x_s, mu_s), 1, seed + 1)[:, 0, :]
    mu_p = _draw_conditional(q_p, x_s, 1, seed + 2)[:, 0, :]
    alpha_p = _draw_conditional(q_n, _qn_context(x_s, mu_p), 1, seed + 3)[:, 0, :]
    x_l = _draw_conditional(q_l, theta_s, 1, seed + 4)[:, 0, :]

    points_s = np.column_stack([mu_s, alpha_s, x_s])
    points_n = np.column_stack([mu_s, alpha_n, x_s])
    points_p = np.column_stack([mu_p, alpha_p, x_s])
    points_l = np.column_stack([mu_s, alpha_s, x_l])
    groups = np.stack([points_s, points_n, points_p, points_l], axis=1).astype(
        np.float32
    )
    return _assert_finite("four-class groups", groups, ndim=3)

four_retry_before = _rqs_retry_count()
four_class_groups = build_four_class_groups(
    N_CLASS, q_p, q_n, q_l, SEED + 620
)
print("Four-class grouped tensor:", four_class_groups.shape)
print(
    "Inverse-RQS kernel calls retried in float64 during four-class construction:",
    _rqs_retry_count() - four_retry_before,
)


## Nuisance-aware structured ratios, normalization, and bridge

The four residuals are

$$r_P=e^a,\quad r_N=e^b,\quad r_L=e^c,\quad r_{PN}=e^{a+b}.$$

Because $a(\mu,x)$ excludes $\alpha$ by architecture, $Z_P$, $Z_N$, and $Z_L$ are the three head-local mass constraints; $Z_{PN}=1$ then follows by iterated expectation and remains a held-out composition diagnostic.

The bridge adds two non-tautological invariances.  At fixed $(\mu,x)$,

$$\ell_m(\alpha)=\log\rho_\alpha+\log q_L-\log q_N+s_N-s_L$$

must equal the nuisance-marginal log likelihood and be independent of $\alpha$.  At fixed $x$,

$$\ell_E(\mu,\alpha)=\log\rho_\mu+\log\rho_\alpha+\log q_L-\log q_P-\log q_N+s_P-s_L$$

must equal the evidence and be independent of both parameters.  We use half the unbiased within-anchor variance of each, with IID component labels for every arithmetic-mixture bridge draw.  The old variance of $r_P$ across $\alpha$ is intentionally absent: with the structured $a$-head it is identically zero and would be a fake constraint.


In [ ]:
def build_four_normalization_bundle(n_groups, n_inner, q_p, q_n, q_l, seed):
    rng = np.random.default_rng(seed)

    # Z_P(x): a(mu,x) does not depend on alpha, so q_N draws are unnecessary.
    theta_zp_anchor = sample_tail_enriched_design(n_groups, rng)
    x_zp = simulate(theta_zp_anchor, rng)
    mu_zp_a = _draw_conditional(q_p, x_zp, n_inner, seed + 1)
    mu_zp_b = _draw_conditional(q_p, x_zp, n_inner, seed + 2)
    x_zp_repeat = np.repeat(x_zp[:, None, :], n_inner, axis=1)
    alpha_filler = np.zeros_like(mu_zp_a)

    # Z_N(x,mu): independent q_N halves at fixed sampled (x,mu).
    theta_zn_anchor = sample_tail_enriched_design(n_groups, rng)
    x_zn = simulate(theta_zn_anchor, rng)
    mu_zn = theta_zn_anchor[:, :1]
    qn_context = _qn_context(x_zn, mu_zn)
    alpha_zn_a = _draw_conditional(q_n, qn_context, n_inner, seed + 3)
    alpha_zn_b = _draw_conditional(q_n, qn_context, n_inner, seed + 4)
    mu_zn_repeat = np.repeat(mu_zn[:, None, :], n_inner, axis=1)
    x_zn_repeat = np.repeat(x_zn[:, None, :], n_inner, axis=1)

    # Z_L(theta): independent q_L halves at fixed sampled theta.
    theta_zl = sample_tail_enriched_design(n_groups, rng)
    x_zl_a = _draw_conditional(q_l, theta_zl, n_inner, seed + 5)
    x_zl_b = _draw_conditional(q_l, theta_zl, n_inner, seed + 6)
    theta_zl_repeat = np.repeat(theta_zl[:, None, :], n_inner, axis=1)

    bundle = {
        "zp_a_points": np.concatenate([mu_zp_a, alpha_filler, x_zp_repeat], axis=2),
        "zp_b_points": np.concatenate([mu_zp_b, alpha_filler, x_zp_repeat], axis=2),
        "zn_a_points": np.concatenate([mu_zn_repeat, alpha_zn_a, x_zn_repeat], axis=2),
        "zn_b_points": np.concatenate([mu_zn_repeat, alpha_zn_b, x_zn_repeat], axis=2),
        "zl_a_points": np.concatenate([theta_zl_repeat, x_zl_a], axis=2),
        "zl_b_points": np.concatenate([theta_zl_repeat, x_zl_b], axis=2),
    }
    for name, value in bundle.items():
        _assert_finite(name, value)
        if len(value) != n_groups:
            raise RuntimeError(f"{name} lost its anchor-group axis.")
    return bundle


def four_normalization_loss(model, bundle, index):
    pairs = [
        ("zp", 1, 2),
        ("zn", 0, 1),
        ("zl", 0, 3),
    ]
    cross_terms, monitor_terms = [], []
    for prefix, numerator, denominator in pairs:
        logits_a = _constraint_logits(model, bundle, f"{prefix}_a_points", index, 4)
        logits_b = _constraint_logits(model, bundle, f"{prefix}_b_points", index, 4)
        cross, monitor = _partition_cross_mass(
            logits_a, logits_b, numerator, denominator
        )
        cross_terms.append(cross)
        monitor_terms.append(monitor)
    return torch.stack(cross_terms).mean(), torch.stack(monitor_terms).mean()


# Four-class banks use four further non-overlapping seed namespaces.
constraints_four_train = [
    build_four_normalization_bundle(
        N_NORM_TRAIN_GROUPS,
        N_NORM_TRAIN_INNER,
        q_p,
        q_n,
        q_l,
        SEED + 60_000 + 100 * bank,
    )
    for bank in range(N_NORM_TRAIN_BANKS)
]
constraints_four_validation = [
    build_four_normalization_bundle(
        N_NORM_VALID_GROUPS,
        N_NORM_VALID_INNER,
        q_p,
        q_n,
        q_l,
        SEED + 70_000 + 100 * bank,
    )
    for bank in range(N_NORM_VALID_BANKS)
]


def build_four_bridge_bundle(n_groups, n_inner, q_p, q_n, q_l, seed):
    if int(n_inner) < 2:
        raise ValueError("Bridge variance requires at least two draws per anchor.")
    rng = np.random.default_rng(seed)

    # Nuisance-marginal invariance at fixed sampled (mu,x).
    theta_anchor = sample_tail_enriched_design(n_groups, rng)
    x_anchor = simulate(theta_anchor, rng)
    mu_anchor = theta_anchor[:, :1]
    alpha = _draw_conditional(
        q_n, _qn_context(x_anchor, mu_anchor), n_inner, seed + 1,
        allocation="iid",
    )
    mu_repeat = np.repeat(mu_anchor[:, None, :], n_inner, axis=1)
    x_repeat = np.repeat(x_anchor[:, None, :], n_inner, axis=1)
    theta_nuisance = np.concatenate([mu_repeat, alpha], axis=2)
    nuisance_points = np.concatenate([theta_nuisance, x_repeat], axis=2).astype(np.float32)
    flat_theta = theta_nuisance.reshape(-1, 2)
    flat_x = x_repeat.reshape(-1, 3)
    flat_mu = mu_repeat.reshape(-1, 1)
    flat_alpha = alpha.reshape(-1, 1)
    nuisance_base = (
        design_alpha_logpdf(flat_alpha[:, 0])
        + _flow_log_prob(q_l, flat_x, context=flat_theta)
        - _flow_log_prob(q_n, flat_alpha, context=_qn_context(flat_x, flat_mu))
    ).reshape(n_groups, n_inner).astype(np.float32)

    # Absolute-evidence invariance at fixed sampled x.
    theta_x = sample_tail_enriched_design(n_groups, rng)
    x_evidence = simulate(theta_x, rng)
    mu = _draw_conditional(
        q_p, x_evidence, n_inner, seed + 2, allocation="iid"
    )
    x_evidence_repeat = np.repeat(x_evidence[:, None, :], n_inner, axis=1)
    flat_mu = mu.reshape(-1, 1)
    flat_x = x_evidence_repeat.reshape(-1, 3)
    alpha = _draw_conditional(
        q_n, _qn_context(flat_x, flat_mu), 1, seed + 3,
        allocation="iid",
    )[:, 0, :].reshape(n_groups, n_inner, 1)
    theta_evidence = np.concatenate([mu, alpha], axis=2)
    evidence_points = np.concatenate([theta_evidence, x_evidence_repeat], axis=2).astype(np.float32)
    flat_theta = theta_evidence.reshape(-1, 2)
    flat_alpha = alpha.reshape(-1, 1)
    evidence_base = (
        design_logpdf(flat_theta)
        + _flow_log_prob(q_l, flat_x, context=flat_theta)
        - _flow_log_prob(q_p, flat_mu, context=flat_x)
        - _flow_log_prob(q_n, flat_alpha, context=_qn_context(flat_x, flat_mu))
    ).reshape(n_groups, n_inner).astype(np.float32)
    return {
        "nuisance_bridge_points": _assert_finite("nuisance bridge points", nuisance_points),
        "nuisance_bridge_base": _assert_finite("nuisance bridge base", nuisance_base),
        "evidence_bridge_points": _assert_finite("evidence bridge points", evidence_points),
        "evidence_bridge_base": _assert_finite("evidence bridge base", evidence_base),
    }


def four_bridge_loss(model, bundle, index):
    nuisance_logits = _constraint_logits(
        model, bundle, "nuisance_bridge_points", index, 4
    )
    nuisance_base = bundle["nuisance_bridge_base"].index_select(0, index).to(device)
    log_marginal = nuisance_base + nuisance_logits[..., 1] - nuisance_logits[..., 3]
    evidence_logits = _constraint_logits(
        model, bundle, "evidence_bridge_points", index, 4
    )
    evidence_base = bundle["evidence_bridge_base"].index_select(0, index).to(device)
    log_evidence = evidence_base + evidence_logits[..., 2] - evidence_logits[..., 3]
    return 0.5 * (
        log_marginal.var(dim=1, unbiased=True).mean()
        + log_evidence.var(dim=1, unbiased=True).mean()
    )


bridge_four_train = [
    build_four_bridge_bundle(
        N_BRIDGE_TRAIN_GROUPS,
        N_BRIDGE_TRAIN_INNER,
        q_p,
        q_n,
        q_l,
        SEED + 80_000 + 100 * bank,
    )
    for bank in range(N_NORM_TRAIN_BANKS)
]
bridge_four_validation = [
    build_four_bridge_bundle(
        N_BRIDGE_VALID_GROUPS,
        N_BRIDGE_VALID_INNER,
        q_p,
        q_n,
        q_l,
        SEED + 90_000 + 100 * bank,
    )
    for bank in range(N_NORM_VALID_BANKS)
]


In [ ]:
four_full = train_multiclass_ensemble(
    four_class_groups,
    n_classes=4,
    checkpoint_dir=MODEL_DIR / "four_class_ce_normalization_bridge",
    seed_base=SEED + 900,
    constraint_train=constraints_four_train,
    constraint_validation=constraints_four_validation,
    constraint_loss_fn=four_normalization_loss,
    lambda_norm=LAMBDA_NORM_4,
    bridge_train=bridge_four_train,
    bridge_validation=bridge_four_validation,
    bridge_loss_fn=four_bridge_loss,
    lambda_bridge=LAMBDA_BRIDGE_4,
)


## Four-class posterior closure

The structured heads provide a conditionally normalized hierarchical posterior,

$$
\widehat p_H(\mu,\alpha\mid x)=
\frac{q_P(\mu\mid x)e^{a(\mu,x)}}{Z_P(x)}
\frac{q_N(\alpha\mid x,\mu)e^{b(\mu,\alpha,x)}}{Z_N(x,\mu)}.
$$

This uses only proposal Monte Carlo normalizers.  We compare it with both the compositionally equivalent population expression $q_Pq_Ne^{a+b}/Z_{PN}$ and the independently normalized likelihood route

$$
\widehat p_L(\mu,\alpha\mid x)\propto
\rho_\mu\rho_\alpha q_L(x\mid\mu,\alpha)e^c/Z_L(\mu,\alpha).
$$

At finite capacity the hierarchical and composed routes can differ; that difference is a diagnostic, not an invitation to multiply in another ad hoc normalization factor.  Analytic contours remain validation-only.


In [ ]:
def four_log_zp(ensemble, x_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    mu = _draw_conditional(q_p, x_values, n_reference, seed)
    alpha_filler = np.zeros_like(mu)
    points = np.concatenate([
        mu, alpha_filler,
        np.repeat(x_values[:, None, :], n_reference, axis=1),
    ], axis=2)
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 1] - logits[..., 2], axis=1) - np.log(n_reference)


def four_log_zl(ensemble, theta_values, n_reference, seed):
    theta_values = np.atleast_2d(np.asarray(theta_values, dtype=np.float32))
    # Stream conditioning contexts.  A paper grid has O(10^4) theta
    # values, so materializing every theta x reference draw at once would
    # create multi-GB ensemble-logit temporaries.
    context_batch = max(1, 65_536 // int(n_reference))
    chunks = []
    for start in range(0, len(theta_values), context_batch):
        theta_chunk = theta_values[start : start + context_batch]
        x = _draw_conditional(
            q_l, theta_chunk, n_reference, seed + start
        )
        points = np.concatenate([
            np.repeat(theta_chunk[:, None, :], n_reference, axis=1), x
        ], axis=2)
        logits = predict_shared_logits(ensemble, points)
        chunks.append(
            logsumexp(logits[..., 0] - logits[..., 3], axis=1)
            - np.log(n_reference)
        )
    return np.concatenate(chunks)

def four_log_zn(ensemble, x_values, mu_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    mu_values = np.asarray(mu_values, dtype=np.float32).reshape(-1, 1)
    if len(x_values) == 1 and len(mu_values) > 1:
        x_values = np.repeat(x_values, len(mu_values), axis=0)
    alpha = _draw_conditional(
        q_n, _qn_context(x_values, mu_values), n_reference, seed
    )
    points = np.concatenate([
        np.repeat(mu_values[:, None, :], n_reference, axis=1),
        alpha,
        np.repeat(x_values[:, None, :], n_reference, axis=1),
    ], axis=2)
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 0] - logits[..., 1], axis=1) - np.log(n_reference)

def four_log_zpn(ensemble, x_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    mu, alpha = _draw_joint_reference(x_values, n_reference, seed)
    points = np.concatenate([
        mu, alpha, np.repeat(x_values[:, None, :], n_reference, axis=1)
    ], axis=2)
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 0] - logits[..., 2], axis=1) - np.log(n_reference)

MU_GRID_2D = np.linspace(-3.5, 3.5, 181 if not SMOKE_MODE else 71)
ALPHA_GRID_2D = np.linspace(-3.2, 3.2, 161 if not SMOKE_MODE else 61)
MU_MESH, ALPHA_MESH = np.meshgrid(MU_GRID_2D, ALPHA_GRID_2D, indexing="ij")
THETA_GRID = np.column_stack([MU_MESH.ravel(), ALPHA_MESH.ravel()])
X_GRID = np.repeat(X_OBS[None, :], len(THETA_GRID), axis=0)
POINTS_GRID = np.column_stack([THETA_GRID, X_GRID]).astype(np.float32)

logits_grid = predict_shared_logits(four_full, POINTS_GRID)

log_qp_grid = _flow_log_prob(
    q_p, THETA_GRID[:, :1], context=X_GRID
)
log_qn_grid = _flow_log_prob(
    q_n,
    THETA_GRID[:, 1:2],
    context=_qn_context(X_GRID, THETA_GRID[:, :1]),
)

log_zp_observed = float(four_log_zp(
    four_full, X_OBS, N_GRID_REFERENCE, SEED + 990
)[0])
log_zn_mu = four_log_zn(
    four_full,
    X_OBS,
    MU_GRID_2D,
    N_GRID_REFERENCE,
    SEED + 991,
)
log_zn_grid = np.repeat(log_zn_mu, len(ALPHA_GRID_2D))
log_joint_hierarchical = (
    log_qp_grid + logits_grid[:, 1] - logits_grid[:, 2] - log_zp_observed
    + log_qn_grid + logits_grid[:, 0] - logits_grid[:, 1] - log_zn_grid
)
posterior_four_hierarchical, _ = normalize_log_surface(
    log_joint_hierarchical.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)

log_zpn_observed = float(four_log_zpn(
    four_full, X_OBS, N_GRID_REFERENCE, SEED + 992
)[0])
log_joint_composed = (
    log_qp_grid + log_qn_grid
    + logits_grid[:, 0] - logits_grid[:, 2] - log_zpn_observed
)
posterior_four_composed, _ = normalize_log_surface(
    log_joint_composed.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)


log_zl_grid = four_log_zl(
    four_full, THETA_GRID, N_GRID_REFERENCE, SEED + 1000
)
log_ql_grid = _flow_log_prob(q_l, X_GRID, context=THETA_GRID)
log_joint_likelihood = (
    design_logpdf(THETA_GRID)
    + log_ql_grid
    + logits_grid[:, 0]
    - logits_grid[:, 3]
    - log_zl_grid
)
posterior_four_likelihood, _ = normalize_log_surface(
    log_joint_likelihood.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)

log_joint_truth = design_logpdf(THETA_GRID) + log_likelihood(X_OBS, THETA_GRID)
posterior_truth_2d, log_evidence_grid_truth = normalize_log_surface(
    log_joint_truth.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)
truth_mu_2d = np.trapezoid(posterior_truth_2d, ALPHA_GRID_2D, axis=1)
truth_alpha_2d = np.trapezoid(posterior_truth_2d, MU_GRID_2D, axis=0)
hierarchical_mu_2d = np.trapezoid(
    posterior_four_hierarchical, ALPHA_GRID_2D, axis=1
)
hierarchical_alpha_2d = np.trapezoid(
    posterior_four_hierarchical, MU_GRID_2D, axis=0
)
composed_mu_2d = np.trapezoid(posterior_four_composed, ALPHA_GRID_2D, axis=1)
composed_alpha_2d = np.trapezoid(posterior_four_composed, MU_GRID_2D, axis=0)
likelihood_mu_2d = np.trapezoid(posterior_four_likelihood, ALPHA_GRID_2D, axis=1)
likelihood_alpha_2d = np.trapezoid(posterior_four_likelihood, MU_GRID_2D, axis=0)
print(
    "Posterior-grid evidence minus wide 1D truth =",
    f"{log_evidence_grid_truth - LOG_EVIDENCE_TRUTH:+.4f}",
    "(finite plotting-window truncation)",
)
four_summary = pd.DataFrame([
    {
        "route": "hierarchically normalized posterior",
        "2D JS distance": js_distance_discrete(
            posterior_truth_2d, posterior_four_hierarchical
        ),
        "mu marginal IAE": integrated_absolute_error(
            truth_mu_2d, hierarchical_mu_2d, MU_GRID_2D
        ),
        "alpha marginal IAE": integrated_absolute_error(
            truth_alpha_2d, hierarchical_alpha_2d, ALPHA_GRID_2D
        ),
    },
    {
        "route": "composed S/P diagnostic",
        "2D JS distance": js_distance_discrete(
            posterior_truth_2d, posterior_four_composed
        ),
        "mu marginal IAE": integrated_absolute_error(
            truth_mu_2d, composed_mu_2d, MU_GRID_2D
        ),
        "alpha marginal IAE": integrated_absolute_error(
            truth_alpha_2d, composed_alpha_2d, ALPHA_GRID_2D
        ),
    },
    {
        "route": "normalized likelihood",
        "2D JS distance": js_distance_discrete(
            posterior_truth_2d, posterior_four_likelihood
        ),
        "mu marginal IAE": integrated_absolute_error(
            truth_mu_2d, likelihood_mu_2d, MU_GRID_2D
        ),
        "alpha marginal IAE": integrated_absolute_error(
            truth_alpha_2d, likelihood_alpha_2d, ALPHA_GRID_2D
        ),
    },
])
display(four_summary.style.format(precision=4))


In [ ]:
def highest_density_levels(density, x_grid, y_grid, masses=(0.5, 0.9)):
    density = np.asarray(density, dtype=float)
    dx = float(np.mean(np.diff(x_grid)))
    dy = float(np.mean(np.diff(y_grid)))
    ordered = np.sort(density.ravel())[::-1]
    cumulative = np.cumsum(ordered) * dx * dy
    return sorted([
        ordered[min(np.searchsorted(cumulative, mass), len(ordered) - 1)]
        for mass in masses
    ])

truth_levels = highest_density_levels(
    posterior_truth_2d, MU_GRID_2D, ALPHA_GRID_2D
)
hierarchical_levels = highest_density_levels(
    posterior_four_hierarchical, MU_GRID_2D, ALPHA_GRID_2D
)
likelihood_levels = highest_density_levels(
    posterior_four_likelihood, MU_GRID_2D, ALPHA_GRID_2D
)

fig, axes = plt.subplots(2, 2, figsize=(11.8, 9.0), constrained_layout=True)
axes[0, 0].contour(
    MU_GRID_2D, ALPHA_GRID_2D, posterior_truth_2d.T,
    levels=truth_levels, colors="black", linewidths=[2.2, 1.5],
)
axes[0, 0].contour(
    MU_GRID_2D, ALPHA_GRID_2D, posterior_four_hierarchical.T,
    levels=hierarchical_levels, colors=FULL4_COLOR, linewidths=[2.2, 1.5], linestyles="--",
)
axes[0, 0].set_title("(a) Joint posterior: hierarchical route\nblack truth; purple learned")

axes[0, 1].contour(
    MU_GRID_2D, ALPHA_GRID_2D, posterior_truth_2d.T,
    levels=truth_levels, colors="black", linewidths=[2.2, 1.5],
)
axes[0, 1].contour(
    MU_GRID_2D, ALPHA_GRID_2D, posterior_four_likelihood.T,
    levels=likelihood_levels, colors=FULL4_COLOR, linewidths=[2.2, 1.5], linestyles="--",
)
axes[0, 1].set_title("(b) Joint posterior: likelihood route\nblack truth; purple learned")
for ax in axes[0]:
    ax.set(xlabel=r"$\mu$", ylabel=r"$\alpha$")
    ax.grid(alpha=0.2)

three_bridge_interp = np.interp(MU_GRID_2D, MU_GRID, closure_bridge["posterior"])
axes[1, 0].plot(MU_GRID_2D, truth_mu_2d, color="black", lw=2.3, label="analytic truth")
axes[1, 0].plot(
    MU_GRID_2D, three_bridge_interp, color=FULL3_COLOR, lw=1.8,
    label="three-class full objective (alpha hidden)",
)
axes[1, 0].plot(
    MU_GRID_2D, hierarchical_mu_2d, color=FULL4_COLOR, lw=2.1,
    label="four-class hierarchical",
)
axes[1, 0].plot(
    MU_GRID_2D, composed_mu_2d, color=FULL4_COLOR, lw=1.5, ls=":",
    label="composed S/P diagnostic",
)
axes[1, 0].plot(
    MU_GRID_2D, likelihood_mu_2d, color=FULL4_COLOR, lw=1.8, ls="--",
    label="four-class likelihood",
)
axes[1, 0].set(xlabel=r"$\mu$", ylabel="marginal posterior", title="(c) Parameter of interest")
axes[1, 0].legend(fontsize=8.5)

axes[1, 1].plot(ALPHA_GRID_2D, truth_alpha_2d, color="black", lw=2.3, label="analytic truth")
axes[1, 1].plot(
    ALPHA_GRID_2D, hierarchical_alpha_2d, color=FULL4_COLOR, lw=2.1,
    label="four-class hierarchical",
)
axes[1, 1].plot(
    ALPHA_GRID_2D, composed_alpha_2d, color=FULL4_COLOR, lw=1.5, ls=":",
    label="composed S/P diagnostic",
)
axes[1, 1].plot(
    ALPHA_GRID_2D, likelihood_alpha_2d, color=FULL4_COLOR, lw=1.8, ls="--",
    label="four-class likelihood",
)
axes[1, 1].set(xlabel=r"$\alpha$", ylabel="marginal posterior", title="(d) Systematic nuisance")
axes[1, 1].legend(fontsize=8.5)
for ax in axes[1]:
    ax.grid(alpha=0.25)

export_exercise9_multiclass_figure(fig, "four_class_joint_posterior_closure")
plt.show()


## Held-out calibration with the nuisance explicit

A single observed-data contour is not an amortized calibration test.  On fresh simulator draws we therefore estimate marginal PIT and equal-tailed coverage for both $\mu$ and $\alpha$.

For every held-out $x$, the calculation draws $\mu\sim q_P$, reweights it by $e^a$, then draws $\alpha\sim q_N$ at each sampled $\mu$ and conditionally reweights by $e^b$.  Normalizing those sample weights within each conditional level implements the hierarchical posterior without evaluating the analytic likelihood, marginal likelihood, or any truth-density residual.  The raw $q_Pq_N$ hierarchy is shown as a proposal baseline.


In [ ]:
def four_class_calibration_pits(
    ensemble,
    theta_true,
    x_true,
    n_mu,
    n_alpha,
    seed,
    context_batch=8,
):
    theta_true = np.asarray(theta_true, dtype=np.float32)
    x_true = np.asarray(x_true, dtype=np.float32)
    learned_mu, learned_alpha = [], []
    proposal_mu, proposal_alpha = [], []
    for start in range(0, len(theta_true), int(context_batch)):
        stop = min(len(theta_true), start + int(context_batch))
        theta_batch = theta_true[start:stop]
        x_batch = x_true[start:stop]
        batch_size = len(x_batch)

        mu = _draw_conditional(q_p, x_batch, n_mu, seed + 10 * start)
        x_at_mu = np.repeat(x_batch[:, None, :], n_mu, axis=1)
        a_points = np.concatenate([mu, np.zeros_like(mu), x_at_mu], axis=2)
        a_logits = predict_shared_logits(ensemble, a_points)
        log_a = a_logits[..., 1] - a_logits[..., 2]
        mu_weights = np.exp(log_a - logsumexp(log_a, axis=1, keepdims=True))
        mu_indicator = mu[..., 0] <= theta_batch[:, 0, None]
        learned_mu.append(np.sum(mu_weights * mu_indicator, axis=1))
        proposal_mu.append(mu_indicator.mean(axis=1))

        flat_mu = mu.reshape(-1, 1)
        flat_x = x_at_mu.reshape(-1, 3)
        alpha = _draw_conditional(
            q_n,
            _qn_context(flat_x, flat_mu),
            n_alpha,
            seed + 10 * start + 1,
        )
        mu_at_alpha = np.repeat(flat_mu[:, None, :], n_alpha, axis=1)
        x_at_alpha = np.repeat(flat_x[:, None, :], n_alpha, axis=1)
        b_points = np.concatenate([mu_at_alpha, alpha, x_at_alpha], axis=2)
        b_logits = predict_shared_logits(ensemble, b_points)
        log_b = b_logits[..., 0] - b_logits[..., 1]
        alpha_weights = np.exp(
            log_b - logsumexp(log_b, axis=1, keepdims=True)
        )
        alpha_truth = np.repeat(theta_batch[:, 1], n_mu)
        alpha_indicator = alpha[..., 0] <= alpha_truth[:, None]
        conditional_cdf = np.sum(alpha_weights * alpha_indicator, axis=1)
        learned_alpha.append(
            np.sum(mu_weights * conditional_cdf.reshape(batch_size, n_mu), axis=1)
        )
        proposal_alpha.append(
            alpha_indicator.reshape(batch_size, n_mu, n_alpha).mean(axis=(1, 2))
        )

    return {
        "proposal q_P q_N": {
            "mu": np.concatenate(proposal_mu),
            "alpha": np.concatenate(proposal_alpha),
        },
        "structured hierarchical": {
            "mu": np.concatenate(learned_mu),
            "alpha": np.concatenate(learned_alpha),
        },
    }


rng = np.random.default_rng(SEED + 1400)
theta_four_calibration = sample_design(N_FOUR_CALIBRATION_CONTEXTS, rng)
x_four_calibration = simulate(theta_four_calibration, rng)
four_pit_values = four_class_calibration_pits(
    four_full,
    theta_four_calibration,
    x_four_calibration,
    N_FOUR_CALIBRATION_MU,
    N_FOUR_CALIBRATION_ALPHA,
    SEED + 1401,
)

FOUR_NOMINAL_COVERAGE = np.linspace(0.05, 0.95, 19)
four_coverage = {
    method: {
        parameter: np.array([
            np.mean(
                (pit >= 0.5 * (1.0 - level))
                & (pit <= 0.5 * (1.0 + level))
            )
            for level in FOUR_NOMINAL_COVERAGE
        ])
        for parameter, pit in values.items()
    }
    for method, values in four_pit_values.items()
}
four_calibration_rows = []
for method, values in four_pit_values.items():
    for parameter, pit in values.items():
        ordered = np.sort(pit)
        uniform = (np.arange(len(ordered)) + 0.5) / len(ordered)
        four_calibration_rows.append({
            "method": method,
            "parameter": parameter,
            "PIT KS distance": float(np.max(np.abs(ordered - uniform))),
            "max coverage error": float(np.max(np.abs(
                four_coverage[method][parameter] - FOUR_NOMINAL_COVERAGE
            ))),
        })
four_calibration_summary = pd.DataFrame(four_calibration_rows)
display(four_calibration_summary.style.format(precision=4))

fig, axes = plt.subplots(2, 2, figsize=(11.0, 8.4), constrained_layout=True)
method_colors = {
    "proposal q_P q_N": "0.55",
    "structured hierarchical": FULL4_COLOR,
}
parameter_symbols = {"mu": r"\mu", "alpha": r"\alpha"}
for column, parameter in enumerate(["mu", "alpha"]):
    symbol = parameter_symbols[parameter]
    for method, values in four_pit_values.items():
        pit = values[parameter]
        ordered = np.sort(pit)
        empirical_cdf = np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
        axes[0, column].plot(
            ordered, empirical_cdf, lw=1.9,
            color=method_colors[method], label=method,
        )
        axes[1, column].plot(
            FOUR_NOMINAL_COVERAGE,
            four_coverage[method][parameter],
            lw=1.9,
            color=method_colors[method],
            label=method,
        )
    axes[0, column].plot([0, 1], [0, 1], color="black", ls="--", lw=1)
    axes[1, column].plot([0, 1], [0, 1], color="black", ls="--", lw=1)
    coverage_sigma = np.sqrt(
        FOUR_NOMINAL_COVERAGE * (1.0 - FOUR_NOMINAL_COVERAGE)
        / N_FOUR_CALIBRATION_CONTEXTS
    )
    axes[1, column].fill_between(
        FOUR_NOMINAL_COVERAGE,
        np.maximum(0.0, FOUR_NOMINAL_COVERAGE - coverage_sigma),
        np.minimum(1.0, FOUR_NOMINAL_COVERAGE + coverage_sigma),
        color="0.7", alpha=0.2, linewidth=0, label=r"$\pm1\sigma$ binomial",
    )
    axes[0, column].set(
        xlabel=rf"${symbol}$ posterior PIT",
        ylabel="empirical CDF",
        title=rf"({chr(97 + column)}) ${symbol}$ PIT",
    )
    axes[1, column].set(
        xlabel="nominal equal-tailed coverage",
        ylabel="empirical coverage",
        title=rf"({chr(99 + column)}) ${symbol}$ coverage",
    )
    for row in range(2):
        axes[row, column].set(xlim=(0, 1), ylim=(0, 1))
        axes[row, column].grid(alpha=0.25)
        axes[row, column].legend(fontsize=8.5)

export_exercise9_multiclass_figure(fig, "four_class_heldout_calibration")
plt.show()


## Fresh fully normalized nuisance and evidence bridges

The bridge was trained using raw evidence-invariance variances, without nested Monte Carlo normalizers.  This section is deliberately different: fresh proposal samples estimate $Z_N$, $Z_L$, and $Z_{PN}$, and the displayed curves add the corresponding $\log Z$ corrections.  They test whether the final CE–normalization–bridge model is Bayes-compatible on held-out anchors.

These curves still do not establish simulator fidelity.  Analytic Gaussian densities place the zero-reference line, and posterior-supported versus extrapolation regions are reported separately.


In [ ]:
def nuisance_consistency_curve(ensemble, mu_value, alpha_grid, x_observed, n_reference, seed):
    alpha_grid = np.asarray(alpha_grid, dtype=float)
    theta = np.column_stack([
        np.full_like(alpha_grid, float(mu_value)), alpha_grid
    ]).astype(np.float32)
    x = np.repeat(np.asarray(x_observed)[None, :], len(theta), axis=0).astype(np.float32)
    points = np.column_stack([theta, x])
    logits = predict_shared_logits(ensemble, points)
    log_ql = _flow_log_prob(q_l, x, context=theta)
    log_qn = _flow_log_prob(
        q_n, theta[:, 1:2], context=_qn_context(x, theta[:, :1])
    )
    log_zn = float(four_log_zn(
        ensemble, x_observed, [mu_value], n_reference, seed
    )[0])
    log_zl = four_log_zl(ensemble, theta, n_reference, seed + 10_000)
    raw = design_alpha_logpdf(alpha_grid) + log_ql - log_qn + logits[:, 1] - logits[:, 3]
    normalized = raw + log_zn - log_zl
    truth = float(marginal_log_likelihood(x_observed, [mu_value])[0])
    return raw - truth, normalized - truth

def evidence_consistency_path(ensemble, theta, x_observed, n_reference, seed):
    theta = np.atleast_2d(np.asarray(theta, dtype=np.float32))
    x = np.repeat(np.asarray(x_observed)[None, :], len(theta), axis=0).astype(np.float32)
    points = np.column_stack([theta, x])
    logits = predict_shared_logits(ensemble, points)
    log_qp = _flow_log_prob(q_p, theta[:, :1], context=x)
    log_qn = _flow_log_prob(
        q_n, theta[:, 1:2], context=_qn_context(x, theta[:, :1])
    )
    log_ql = _flow_log_prob(q_l, x, context=theta)
    log_zpn = float(four_log_zpn(
        ensemble, x_observed, n_reference, seed
    )[0])
    log_zl = four_log_zl(ensemble, theta, n_reference, seed + 10_000)
    raw = (
        design_logpdf(theta) + log_ql - log_qp - log_qn
        + logits[:, 2] - logits[:, 3]
    )
    normalized = raw + log_zpn - log_zl
    return raw - LOG_EVIDENCE_TRUTH, normalized - LOG_EVIDENCE_TRUTH

# Locate the two most separated posterior modes for illustrative slices.
local_peak = np.r_[False, (truth_mu[1:-1] > truth_mu[:-2]) & (truth_mu[1:-1] > truth_mu[2:]), False]
peak_candidates = MU_GRID[local_peak]
peak_heights = truth_mu[local_peak]
if len(peak_candidates) >= 2:
    selected = np.argsort(peak_heights)[-2:]
    MU_SLICES = np.sort(peak_candidates[selected])
else:
    MU_SLICES = np.array([-1.4, 1.2])

ALPHA_CONSISTENCY_GRID = np.linspace(-2.6, 2.6, 81 if not SMOKE_MODE else 31)
nuisance_curves = {}
for index, mu_value in enumerate(MU_SLICES):
    nuisance_curves[float(mu_value)] = nuisance_consistency_curve(
        four_full,
        mu_value,
        ALPHA_CONSISTENCY_GRID,
        X_OBS,
        N_DIAGNOSTIC_REFERENCE,
        SEED + 1100 + 10 * index,
    )

MU_PATH = np.linspace(-3.0, 3.0, 101 if not SMOKE_MODE else 41)
THETA_MU_PATH = np.column_stack([MU_PATH, np.zeros_like(MU_PATH)])
raw_mu_path, normalized_mu_path = evidence_consistency_path(
    four_full, THETA_MU_PATH, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 1150
)
ALPHA_PATH = np.linspace(-2.6, 2.6, 101 if not SMOKE_MODE else 41)
mu_mode = float(MU_GRID[np.argmax(truth_mu)])
THETA_ALPHA_PATH = np.column_stack([
    np.full_like(ALPHA_PATH, mu_mode), ALPHA_PATH
])
raw_alpha_path, normalized_alpha_path = evidence_consistency_path(
    four_full, THETA_ALPHA_PATH, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 1160
)

# Raw bridge variance on disjoint validation banks (no analytic density).
four_bridge_rows = []
for bank_index, bundle in enumerate(bridge_four_validation):
    nuisance_logits = predict_shared_logits(four_full, bundle["nuisance_bridge_points"])
    nuisance_implied = (
        bundle["nuisance_bridge_base"] + nuisance_logits[..., 1] - nuisance_logits[..., 3]
    )
    evidence_logits = predict_shared_logits(four_full, bundle["evidence_bridge_points"])
    evidence_implied = (
        bundle["evidence_bridge_base"] + evidence_logits[..., 2] - evidence_logits[..., 3]
    )
    for label, values in [
        ("nuisance-marginal bridge", nuisance_implied),
        ("absolute-evidence bridge", evidence_implied),
    ]:
        rms = np.std(values, axis=1, ddof=1)
        four_bridge_rows.append({
            "bank": bank_index,
            "bridge": label,
            "median within-anchor RMS": float(np.median(rms)),
            "q95 within-anchor RMS": float(np.quantile(rms, 0.95)),
            "max within-anchor RMS": float(np.max(rms)),
        })
four_bridge_validation_summary = pd.DataFrame(four_bridge_rows)
display(four_bridge_validation_summary.style.format(precision=4))


# Held-out Z distributions from ordinary, fresh design anchors.
rng = np.random.default_rng(SEED + 1190)
n_z_contexts = max(48, n_context_check)
theta_z_check = sample_design(n_z_contexts, rng)
x_z_check = simulate(theta_z_check, rng)
n_z_reference = min(256, N_DIAGNOSTIC_REFERENCE)
heldout_four_z = {
    r"$Z_P$": four_log_zp(
        four_full, x_z_check, n_z_reference, SEED + 1191
    ),
    r"$Z_N$": four_log_zn(
        four_full,
        x_z_check,
        theta_z_check[:, 0],
        n_z_reference,
        SEED + 1192,
    ),
    r"$Z_L$": four_log_zl(
        four_full, theta_z_check, n_z_reference, SEED + 1193
    ),
    r"$Z_{PN}$": four_log_zpn(
        four_full, x_z_check, n_z_reference, SEED + 1194
    ),
}
for name, values in heldout_four_z.items():
    print(
        f"{name:8s}: median|log Z|={np.median(np.abs(values)):.3f}, "
        f"q95={np.quantile(np.abs(values), .95):.3f}, "
        f"max={np.max(np.abs(values)):.3f}, "
        f"RMS={np.sqrt(np.mean(values**2)):.3f}"
    )


n_tail_four = 2_000 if SMOKE_MODE else (20_000 if FAST_MODE else 80_000)
mu_joint_tail, alpha_joint_tail = _draw_joint_reference(
    X_OBS[None, :], n_tail_four, SEED + 101_191
)
joint_tail_points = np.concatenate([
    mu_joint_tail,
    alpha_joint_tail,
    np.repeat(X_OBS[None, None, :], n_tail_four, axis=1),
], axis=2)[0]
joint_tail_logits = predict_shared_logits(four_full, joint_tail_points)

alpha_n_tail = _draw_conditional(
    q_n,
    _qn_context(X_OBS, [mu_mode]),
    n_tail_four,
    SEED + 101_192,
)[0]
nuisance_tail_points = np.column_stack([
    np.full(n_tail_four, mu_mode),
    alpha_n_tail[:, 0],
    np.repeat(X_OBS[None, :], n_tail_four, axis=0),
])
nuisance_tail_logits = predict_shared_logits(four_full, nuisance_tail_points)

theta_mode = THETA_GRID[int(np.argmax(posterior_truth_2d))]
x_l_tail = _draw_conditional(
    q_l, theta_mode[None, :], n_tail_four, SEED + 101_193
)[0]
likelihood_tail_points = np.column_stack([
    np.repeat(theta_mode[None, :], n_tail_four, axis=0), x_l_tail
])
likelihood_tail_logits = predict_shared_logits(four_full, likelihood_tail_points)

four_tail_log_ratios = {
    r"$r_P=N/P$": joint_tail_logits[:, 1] - joint_tail_logits[:, 2],
    r"$r_N=S/N$": nuisance_tail_logits[:, 0] - nuisance_tail_logits[:, 1],
    r"$r_L=S/L$": likelihood_tail_logits[:, 0] - likelihood_tail_logits[:, 3],
    r"$r_{PN}=S/P$": joint_tail_logits[:, 0] - joint_tail_logits[:, 2],
}
four_tail_rows = []
for ratio_name, log_ratio in four_tail_log_ratios.items():
    summary = importance_tail_summary(log_ratio)
    four_tail_rows.append({
        "partition ratio": ratio_name,
        "log mean ratio": float(logsumexp(log_ratio) - np.log(len(log_ratio))),
        "ESS fraction": summary["ESS_fraction"],
        "Pareto k": summary["pareto_k"],
        "max weight": summary["max_weight_fraction"],
    })
four_tail_summary = pd.DataFrame(four_tail_rows)
display(four_tail_summary.style.format(precision=4))

four_head_saturation = pd.DataFrame([
    {
        "head / proposal": "a = log(N/P), joint q_P q_N",
        "saturation fraction": head_saturation_fraction(
            four_full, joint_tail_points, "a"
        ),
    },
    {
        "head / proposal": "b = log(S/N), q_N",
        "saturation fraction": head_saturation_fraction(
            four_full, nuisance_tail_points, "b"
        ),
    },
    {
        "head / proposal": "c = log(S/L), q_L",
        "saturation fraction": head_saturation_fraction(
            four_full, likelihood_tail_points, "c"
        ),
    },
])
display(four_head_saturation.style.format(precision=6))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.7), constrained_layout=True)
colors = ["#CC79A7", FULL4_COLOR]
for color, (mu_value, (raw, normalized)) in zip(colors, nuisance_curves.items()):
    axes[0, 0].plot(
        ALPHA_CONSISTENCY_GRID, raw, color=color, lw=1.2, ls=":",
        label=rf"raw, $\mu={mu_value:+.2f}$",
    )
    axes[0, 0].plot(
        ALPHA_CONSISTENCY_GRID, normalized, color=color, lw=2.0,
        label=rf"normalized, $\mu={mu_value:+.2f}$",
    )
axes[0, 0].axhline(0.0, color="black", lw=1)
axes[0, 0].set(
    xlabel=r"$\alpha$ used in the reconstruction",
    ylabel=r"$\log\widehat p_m-\log p_m^{\rm true}$",
    title="(a) Nuisance-marginal consistency",
)
axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(MU_PATH, raw_mu_path, color="0.55", lw=1.2, ls=":", label="raw")
axes[0, 1].plot(MU_PATH, normalized_mu_path, color=FULL4_COLOR, lw=2.0, label="normalized")
axes[0, 1].axhline(0.0, color="black", lw=1)
axes[0, 1].set(
    xlabel=r"$\mu$ at $\alpha=0$", ylabel=r"$\log\widehat m-\log m^{\rm true}$",
    title="(b) Evidence consistency along a POI path",
)
axes[0, 1].legend(fontsize=8.5)

axes[1, 0].plot(ALPHA_PATH, raw_alpha_path, color="0.55", lw=1.2, ls=":", label="raw")
axes[1, 0].plot(ALPHA_PATH, normalized_alpha_path, color=FULL4_COLOR, lw=2.0, label="normalized")
axes[1, 0].axhline(0.0, color="black", lw=1)
axes[1, 0].set(
    xlabel=rf"$\alpha$ at $\mu={mu_mode:+.2f}$",
    ylabel=r"$\log\widehat m-\log m^{\rm true}$",
    title="(c) Evidence consistency along a nuisance path",
)
axes[1, 0].legend(fontsize=8.5)

for label, values in heldout_four_z.items():
    ordered = np.sort(values)
    cdf = np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
    axes[1, 1].plot(ordered, cdf, lw=1.8, label=label)
axes[1, 1].axvline(0.0, color="black", lw=1)
axes[1, 1].set(
    xlabel=r"held-out $\log Z$", ylabel="empirical CDF",
    title="(d) Four conditional partitions",
)
axes[1, 1].legend(fontsize=8.5)
for ax in axes.flat:
    ax.grid(alpha=0.25)

export_exercise9_multiclass_figure(fig, "four_class_consistency_and_normalization")
plt.show()


## A systematic-prior and auxiliary-measurement update

Because the four-class model retains $\alpha$, the same trained posterior can be updated without retraining.  We replace the nuisance design density by
$\pi_{\rm new}(\alpha)=\mathcal N(0.30,0.45^2)$ and include an auxiliary measurement
$a_{\rm obs}=0.10$ with $a\mid\alpha\sim\mathcal N(\alpha,0.25^2)$.

On the grid the learned joint density is multiplied by

$$
  \frac{\pi_{\rm new}(\alpha)}{\rho_\alpha(\alpha)}p(a_{\rm obs}\mid\alpha).
$$

This cell is an application check, not part of classifier training.


In [ ]:
ALPHA_PRIOR_MEAN, ALPHA_PRIOR_SIGMA = 0.30, 0.45
A_OBSERVED, SIGMA_A = 0.10, 0.25
log_update_factor = (
    norm.logpdf(
        THETA_GRID[:, 1], loc=ALPHA_PRIOR_MEAN, scale=ALPHA_PRIOR_SIGMA
    )
    - design_alpha_logpdf(THETA_GRID[:, 1])
    + norm.logpdf(A_OBSERVED, loc=THETA_GRID[:, 1], scale=SIGMA_A)
).reshape(MU_MESH.shape)
learned_updated, _ = normalize_log_surface(
    np.log(np.maximum(posterior_four_hierarchical, 1.0e-300)) + log_update_factor,
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
truth_updated, _ = normalize_log_surface(
    log_joint_truth.reshape(MU_MESH.shape) + log_update_factor,
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
learned_update_mu = np.trapezoid(learned_updated, ALPHA_GRID_2D, axis=1)
learned_update_alpha = np.trapezoid(learned_updated, MU_GRID_2D, axis=0)
truth_update_mu = np.trapezoid(truth_updated, ALPHA_GRID_2D, axis=1)
truth_update_alpha = np.trapezoid(truth_updated, MU_GRID_2D, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11.3, 4.3), constrained_layout=True)
axes[0].plot(MU_GRID_2D, truth_update_mu, color="black", lw=2.3, label="updated truth")
axes[0].plot(MU_GRID_2D, learned_update_mu, color=FULL4_COLOR, lw=2.0, label="updated hierarchical four-class")
axes[0].plot(MU_GRID_2D, truth_mu_2d, color="0.6", ls="--", label="baseline truth")
axes[0].set(xlabel=r"$\mu$", ylabel="posterior density", title="(a) Updated POI")
axes[1].plot(ALPHA_GRID_2D, truth_update_alpha, color="black", lw=2.3, label="updated truth")
axes[1].plot(ALPHA_GRID_2D, learned_update_alpha, color=FULL4_COLOR, lw=2.0, label="updated hierarchical four-class")
axes[1].plot(ALPHA_GRID_2D, truth_alpha_2d, color="0.6", ls="--", label="baseline truth")
axes[1].set(xlabel=r"$\alpha$", ylabel="posterior density", title="(b) Updated systematic")
for ax in axes:
    ax.legend(fontsize=8.5)
    ax.grid(alpha=0.25)
export_exercise9_multiclass_figure(fig, "four_class_systematic_update")
plt.show()

print(
    "Total inverse-RQS kernel calls retried in float64 in this run:",
    _rqs_retry_count(),
)


# Table-2 applications of the trained dual model

The final four-class model now supplies both normalized posterior and normalized generative-likelihood routes.  The following demonstrations use only the frozen proposal mixtures, learned ratios, and known design/prior densities:

1. **corrected likelihood generation** by sampling-importance-resampling from $q_L$ with weights $e^c/Z_L$;
2. **absolute evidence** by importance integration of the corrected likelihood over $q_Pq_N$;
3. **posterior-predictive generation** through both the hNPE and hNDE routes;
4. **selection integrals** from a pre-selection likelihood surrogate.

The analytic Gaussian simulator is called only to create validation samples/curves.  Importance ESS, Pareto-$k$, and maximum weight are part of the result, not optional decoration.


In [ ]:
N_APPLICATION = 256 if SMOKE_MODE else (2_048 if FAST_MODE else 4_096)
N_APPLICATION_Z = 32 if SMOKE_MODE else (64 if FAST_MODE else 128)
N_GENERATIVE = 512 if SMOKE_MODE else (5_000 if FAST_MODE else 20_000)


def corrected_likelihood_candidates(theta, n_candidates, seed):
    theta = np.atleast_2d(np.asarray(theta, dtype=np.float32))
    x = _draw_conditional(q_l, theta, n_candidates, seed)
    theta_repeat = np.repeat(theta[:, None, :], n_candidates, axis=1)
    points = np.concatenate([theta_repeat, x], axis=2)
    logits = predict_shared_logits(four_full, points)
    log_weight = logits[..., 0] - logits[..., 3]
    log_z = logsumexp(log_weight, axis=1) - np.log(n_candidates)
    weight = np.exp(log_weight - logsumexp(log_weight, axis=1, keepdims=True))
    return x, weight, log_z


def importance_resample(values, weights, seed):
    values = np.asarray(values)
    weights = np.asarray(weights, dtype=float)
    rng = np.random.default_rng(seed)
    index = rng.choice(len(values), size=len(values), replace=True, p=weights / weights.sum())
    return values[index]


APP_THETA = np.array([
    [float(MU_GRID[np.argmax(truth_mu)]), 0.0],
    [-1.35, 0.75],
    [2.10, -0.60],
], dtype=np.float32)
rng = np.random.default_rng(SEED + 1500)
generative_rows = []
proposal_generations = []
corrected_generations = []
truth_generations = []
for index, theta in enumerate(APP_THETA):
    candidates, weights, log_z = corrected_likelihood_candidates(
        theta[None, :], N_GENERATIVE, SEED + 1501 + index
    )
    corrected = importance_resample(candidates[0], weights[0], SEED + 1510 + index)
    truth = simulate(np.repeat(theta[None, :], N_GENERATIVE, axis=0), rng)
    proposal_generations.append(candidates[0])
    corrected_generations.append(corrected)
    truth_generations.append(truth)
    tail = importance_tail_summary(np.log(np.maximum(weights[0], 1e-300)))
    generative_rows.append({
        "theta": tuple(theta),
        "log Z_L": float(log_z[0]),
        "proposal mean coordinate W1": float(np.mean([
            wasserstein_distance(candidates[0, :, j], truth[:, j])
            for j in range(3)
        ])),
        "corrected mean coordinate W1": float(np.mean([
            wasserstein_distance(corrected[:, j], truth[:, j])
            for j in range(3)
        ])),
        "ESS fraction": tail["ESS_fraction"],
        "Pareto k": tail["pareto_k"],
        "max weight": tail["max_weight_fraction"],
    })
display(pd.DataFrame(generative_rows).style.format(precision=4))


## Absolute evidence and posterior-predictive generation

With $g(\mu,\alpha\mid x_o)=q_P(\mu\mid x_o)q_N(\alpha\mid x_o,\mu)$ and

$$\widehat L(x_o\mid\theta)=q_L(x_o\mid\theta)\,e^{c(\theta,x_o)}/\widehat Z_L(\theta),$$

the absolute evidence is a **nested Monte Carlo** importance mean of $\rho(\theta)\widehat L/g$.  Because $\widehat L$ contains $1/\widehat Z_L$, finite inner banks have reciprocal-estimator bias.  We therefore show both independent outer-pool convergence and an explicit inner-$N_Z$ scan; an outer ESS alone cannot diagnose this bias.  For predictive generation we draw one $x^{\rm rep}\sim q_L(\cdot\mid\theta)$ and multiply by its likelihood-correction weight.  We construct parameter weights once through the posterior head and once through the likelihood/evidence route; the bridge predicts that their predictive distributions agree.


In [ ]:
mu_app, alpha_app = _draw_joint_reference(
    X_OBS[None, :], N_APPLICATION, SEED + 1530
)
theta_app = np.concatenate([mu_app, alpha_app], axis=2)[0].astype(np.float32)
x_obs_app = np.repeat(X_OBS[None, :], N_APPLICATION, axis=0).astype(np.float32)
points_obs_app = np.column_stack([theta_app, x_obs_app])
logits_obs_app = predict_shared_logits(four_full, points_obs_app)
log_qp_app = _flow_log_prob(q_p, theta_app[:, :1], context=x_obs_app)
log_qn_app = _flow_log_prob(
    q_n, theta_app[:, 1:2], context=_qn_context(x_obs_app, theta_app[:, :1])
)
log_ql_app = _flow_log_prob(q_l, x_obs_app, context=theta_app)
log_zl_app = four_log_zl(
    four_full, theta_app, N_APPLICATION_Z, SEED + 11_530
)
log_likelihood_app = log_ql_app + logits_obs_app[:, 0] - logits_obs_app[:, 3] - log_zl_app
log_evidence_weight = design_logpdf(theta_app) + log_likelihood_app - log_qp_app - log_qn_app

def absolute_evidence_log_weight_pool(n_outer, n_inner, outer_seed, inner_seed=None):
    if inner_seed is None:
        inner_seed = int(outer_seed) + 10_000
    mu, alpha = _draw_joint_reference(X_OBS[None, :], n_outer, outer_seed)
    theta = np.concatenate([mu, alpha], axis=2)[0].astype(np.float32)
    x = np.repeat(X_OBS[None, :], n_outer, axis=0).astype(np.float32)
    points = np.column_stack([theta, x])
    logits = predict_shared_logits(four_full, points)
    log_qp = _flow_log_prob(q_p, theta[:, :1], context=x)
    log_qn = _flow_log_prob(
        q_n, theta[:, 1:2], context=_qn_context(x, theta[:, :1])
    )
    log_ql = _flow_log_prob(q_l, x, context=theta)
    log_zl = four_log_zl(four_full, theta, n_inner, inner_seed)
    return (
        design_logpdf(theta) + log_ql + logits[:, 0] - logits[:, 3]
        - log_zl - log_qp - log_qn
    )


# Each row below is a genuinely independent outer pool with a fresh inner-Z
# bank. Prefixes within a row form one convergence curve; bands are across
# independent rows, not permutations of one shared pool.
EVIDENCE_SIZES = np.unique(np.geomspace(32, N_APPLICATION, 8).astype(int))
N_EVIDENCE_REPEATS = 3 if SMOKE_MODE else (6 if FAST_MODE else 8)
evidence_traces = []
for repeat in range(N_EVIDENCE_REPEATS):
    repeat_log_weight = absolute_evidence_log_weight_pool(
        N_APPLICATION, N_APPLICATION_Z,
        SEED + 1_600 + 100 * repeat,
        SEED + 11_600 + 100 * repeat,
    )
    evidence_traces.append([
        float(logsumexp(repeat_log_weight[:n]) - np.log(n))
        for n in EVIDENCE_SIZES
    ])
evidence_traces = np.asarray(evidence_traces)

# Finite-inner reciprocal bias is assessed with one fixed outer pool per
# repeat and a separate inner-Z stream for every inner-bank size.
EVIDENCE_INNER_SIZES = np.array(
    [8, 16, 32] if SMOKE_MODE else ([16, 32, 64, 128] if FAST_MODE else [32, 64, 128, 256, 512])
)
N_EVIDENCE_INNER_OUTER = min(N_APPLICATION, 256 if SMOKE_MODE else (512 if FAST_MODE else 1_024))
N_EVIDENCE_INNER_REPEATS = 3 if SMOKE_MODE else (4 if FAST_MODE else 6)
evidence_inner_scan = np.empty((N_EVIDENCE_INNER_REPEATS, len(EVIDENCE_INNER_SIZES)))
for repeat in range(N_EVIDENCE_INNER_REPEATS):
    for inner_index, n_inner in enumerate(EVIDENCE_INNER_SIZES):
        scan_log_weight = absolute_evidence_log_weight_pool(
            N_EVIDENCE_INNER_OUTER,
            int(n_inner),
            SEED + 2_600 + 100 * repeat,
            SEED + 12_600 + 100 * repeat + 10 * inner_index,
        )
        evidence_inner_scan[repeat, inner_index] = (
            logsumexp(scan_log_weight) - np.log(len(scan_log_weight))
        )

evidence_tail = importance_tail_summary(log_evidence_weight)
print(
    f"absolute log evidence: learned={logsumexp(log_evidence_weight)-np.log(N_APPLICATION):.5f}, "
    f"truth={LOG_EVIDENCE_TRUTH:.5f}; ESS fraction={evidence_tail['ESS_fraction']:.4f}, "
    f"Pareto k={evidence_tail['pareto_k']:.3f}, max weight={evidence_tail['max_weight_fraction']:.4f}"
)

# One corrected likelihood draw per theta, followed by route-specific weighting.
x_rep = _draw_conditional(q_l, theta_app, 1, SEED + 1540)[:, 0, :]
rep_points = np.column_stack([theta_app, x_rep])
rep_logits = predict_shared_logits(four_full, rep_points)
log_rep_correction = rep_logits[:, 0] - rep_logits[:, 3] - log_zl_app
log_theta_hnpe = logits_obs_app[:, 0] - logits_obs_app[:, 2]
log_predictive_hnpe = log_theta_hnpe + log_rep_correction
log_predictive_hnde = log_evidence_weight + log_rep_correction

def normalized_exp(log_weight):
    return np.exp(log_weight - logsumexp(log_weight))

predictive_hnpe = importance_resample(
    x_rep, normalized_exp(log_predictive_hnpe), SEED + 1541
)
predictive_hnde = importance_resample(
    x_rep, normalized_exp(log_predictive_hnde), SEED + 1542
)
rng_truth = np.random.default_rng(SEED + 1543)
truth_index = rng_truth.choice(
    len(THETA_GRID), size=N_APPLICATION, replace=True,
    p=posterior_truth_2d.ravel() / posterior_truth_2d.sum(),
)
predictive_truth = simulate(THETA_GRID[truth_index], rng_truth)

predictive_rows = []
for name, values, log_weight in [
    ("hNPE route", predictive_hnpe, log_predictive_hnpe),
    ("hNDE route", predictive_hnde, log_predictive_hnde),
]:
    tail = importance_tail_summary(log_weight)
    predictive_rows.append({
        "route": name,
        "mean coordinate W1": float(np.mean([
            wasserstein_distance(values[:, j], predictive_truth[:, j]) for j in range(3)
        ])),
        "ESS fraction": tail["ESS_fraction"],
        "Pareto k": tail["pareto_k"],
        "max weight": tail["max_weight_fraction"],
    })
display(pd.DataFrame(predictive_rows).style.format(precision=4))

fig, axes = plt.subplots(1, 3, figsize=(16.4, 4.4), constrained_layout=True)
for index, (proposal, corrected, truth) in enumerate(
    zip(proposal_generations, corrected_generations, truth_generations)
):
    axes[0].hist(
        truth[:, 1], bins=55, density=True, histtype="step", lw=1.8,
        label=rf"truth $\theta_{index+1}$",
    )
    axes[0].hist(
        proposal[:, 1], bins=55, density=True, histtype="step", lw=1.1,
        color=f"C{index}", alpha=0.65, label=rf"$q_L$, $\theta_{index+1}$",
    )
    axes[0].hist(
        corrected[:, 1], bins=55, density=True, histtype="step", lw=1.5,
        ls="--", color=f"C{index}", label=rf"corrected, $\theta_{index+1}$",
    )
axes[0].set(xlabel=r"$x_2$", ylabel="density", title="(a) Normalized generative likelihood")
axes[0].legend(fontsize=7.0, ncol=3)

median = np.median(evidence_traces, axis=0)
low, high = np.quantile(evidence_traces, [0.16, 0.84], axis=0)
axes[1].plot(EVIDENCE_SIZES, median, marker="o", color=FULL4_COLOR, label="learned")
axes[1].fill_between(EVIDENCE_SIZES, low, high, color=FULL4_COLOR, alpha=0.22)
axes[1].axhline(LOG_EVIDENCE_TRUTH, color="black", ls="--", label="analytic truth")
axes[1].set(xscale="log", xlabel="outer importance samples", ylabel="log evidence", title="(b) Absolute evidence")
axes[1].legend(fontsize=8)

inner_median = np.median(evidence_inner_scan, axis=0)
inner_low, inner_high = np.quantile(evidence_inner_scan, [0.16, 0.84], axis=0)
axes[2].plot(EVIDENCE_INNER_SIZES, inner_median, marker="o", color=FULL4_COLOR, label="nested MC")
axes[2].fill_between(EVIDENCE_INNER_SIZES, inner_low, inner_high, color=FULL4_COLOR, alpha=0.22)
axes[2].axhline(LOG_EVIDENCE_TRUTH, color="black", ls="--", label="analytic truth")
axes[2].set(xscale="log", xlabel=r"inner samples for $\widehat Z_L$", ylabel="log evidence", title="(c) Reciprocal-bias scan")
axes[2].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.2)
export_exercise9_multiclass_figure(fig, "table2_likelihood_generation_and_nested_evidence")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.1), constrained_layout=True)
for coordinate, ax in enumerate(axes):
    ax.hist(predictive_truth[:, coordinate], bins=50, density=True, histtype="step", color="black", lw=2.0, label="analytic-grid truth")
    ax.hist(predictive_hnpe[:, coordinate], bins=50, density=True, histtype="step", color=FULL3_COLOR, lw=1.7, label="hNPE route")
    ax.hist(predictive_hnde[:, coordinate], bins=50, density=True, histtype="step", color=FULL4_COLOR, lw=1.7, ls="--", label="hNDE route")
    ax.axvline(X_OBS[coordinate], color="0.5", ls=":", label="observed")
    ax.set(xlabel=rf"$x_{coordinate+1}^{{\rm rep}}$", ylabel="density", title=f"({chr(97+coordinate)}) Posterior predictive")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=7.5)
export_exercise9_multiclass_figure(fig, "table2_posterior_predictive_routes")
plt.show()


## Selection integrals without the simulator

Define the retained pre-selection statistic $T(x)=x_2-0.6x_3$ and selection $T>0.5$.  For proposal draws $x_k\sim q_L(\cdot\mid\mu,\alpha)$,

$$\widehat\beta(\mu,\alpha)=\frac{\sum_k\mathbf1[T(x_k)>0.5]e^{c_k}}{\sum_ke^{c_k}}.$$

This requires a surrogate for the **pre-selection** distribution and retention of the selection statistic.  A model trained only on detected events cannot recover the missing efficiency.  The Gaussian analytic efficiency below is validation-only.


In [ ]:
N_SELECTION_GRID = 13 if SMOKE_MODE else (25 if FAST_MODE else 41)
N_SELECTION_DRAWS = 128 if SMOKE_MODE else (256 if FAST_MODE else 512)
MU_SELECTION = np.linspace(-3.2, 3.2, N_SELECTION_GRID)
ALPHA_SELECTION = np.linspace(-2.4, 2.4, N_SELECTION_GRID)
MU_SEL_MESH, ALPHA_SEL_MESH = np.meshgrid(MU_SELECTION, ALPHA_SELECTION, indexing="ij")
THETA_SELECTION = np.column_stack([MU_SEL_MESH.ravel(), ALPHA_SEL_MESH.ravel()]).astype(np.float32)
def learned_selection_efficiency(theta, n_draws, seed):
    theta = np.atleast_2d(np.asarray(theta, dtype=np.float32))
    x = _draw_conditional(q_l, theta, n_draws, seed)
    theta_repeat = np.repeat(theta[:, None, :], n_draws, axis=1)
    points = np.concatenate([theta_repeat, x], axis=2)
    logits = predict_shared_logits(four_full, points)
    log_weight = logits[..., 0] - logits[..., 3]
    weight = np.exp(log_weight - np.max(log_weight, axis=1, keepdims=True))
    selected = (x[..., 1] - 0.6 * x[..., 2]) > 0.5
    return np.sum(weight * selected, axis=1) / np.sum(weight, axis=1)


beta_learned = learned_selection_efficiency(
    THETA_SELECTION, N_SELECTION_DRAWS, SEED + 201_560
).reshape(MU_SEL_MESH.shape)

mean_t = 0.72 * MU_SEL_MESH**2 - 0.48 * np.cos(MU_SEL_MESH) - 0.58 * ALPHA_SEL_MESH
sigma_t = np.sqrt(SIMULATOR_SIGMA[1] ** 2 + 0.6**2 * SIMULATOR_SIGMA[2] ** 2)
beta_truth = 1.0 - norm.cdf((0.5 - mean_t) / sigma_t)

# Full-support population integrals use direct prior Monte Carlo rather than
# pretending that the displayed heat-map window covers the broad mixtures.
N_SELECTION_POP = 128 if SMOKE_MODE else (512 if FAST_MODE else 1_024)
rng_population = np.random.default_rng(SEED + 201_561)
theta_population_design = sample_design(N_SELECTION_POP, rng_population)
theta_population_updated = np.column_stack([
    sample_mu(N_SELECTION_POP, rng_population),
    rng_population.normal(ALPHA_PRIOR_MEAN, ALPHA_PRIOR_SIGMA, N_SELECTION_POP),
]).astype(np.float32)

def analytic_selection_efficiency(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    mean = 0.72 * theta[:, 0] ** 2 - 0.48 * np.cos(theta[:, 0]) - 0.58 * theta[:, 1]
    return 1.0 - norm.cdf((0.5 - mean) / sigma_t)

for label, theta, seed in [
    ("design prior", theta_population_design, SEED + 201_562),
    ("updated nuisance prior", theta_population_updated, SEED + 201_563),
]:
    learned_population = learned_selection_efficiency(
        theta, N_SELECTION_DRAWS, seed
    )
    truth_population = analytic_selection_efficiency(theta)
    print(
        f"population selection, {label}: learned={learned_population.mean():.5f}, "
        f"truth={truth_population.mean():.5f}, "
        f"MC paired SE={np.std(learned_population-truth_population, ddof=1)/np.sqrt(len(theta)):.5f}"
    )

fig, axes = plt.subplots(1, 3, figsize=(13.7, 4.2), constrained_layout=True)
extent = [ALPHA_SELECTION[0], ALPHA_SELECTION[-1], MU_SELECTION[0], MU_SELECTION[-1]]
im0 = axes[0].imshow(beta_truth, origin="lower", aspect="auto", extent=extent, vmin=0, vmax=1, cmap="viridis")
im1 = axes[1].imshow(beta_learned, origin="lower", aspect="auto", extent=extent, vmin=0, vmax=1, cmap="viridis")
residual = beta_learned - beta_truth
bound = max(0.02, float(np.max(np.abs(residual))))
im2 = axes[2].imshow(residual, origin="lower", aspect="auto", extent=extent, vmin=-bound, vmax=bound, cmap="coolwarm")
axes[0].set_title("(a) Analytic validation truth")
axes[1].set_title("(b) Corrected hNDE")
axes[2].set_title("(c) Learned minus truth")
for ax in axes:
    ax.set(xlabel=r"$\alpha$", ylabel=r"$\mu$")
fig.colorbar(im0, ax=axes[:2], label=r"selection efficiency $\beta$")
fig.colorbar(im2, ax=axes[2], label="efficiency residual")
export_exercise9_multiclass_figure(fig, "table2_selection_integral")
plt.show()


## Conclusions and paper-run checklist

This version tests the full dual construction rather than a post-hoc bridge check:

1. all structured heads have binary-corrector-scale capacity, while vector proposal flows use learned LU mixing and every proposal is an explicit arithmetic density mixture;
2. the controlled Part-I comparison is CE, CE–normalization, and CE–normalization–bridge on common sampled data and seeds;
3. the bridge is the variance of a physically reconstructed evidence, not a tautological logit cycle, and noisy nested $\log Z$ estimates never enter its gradient;
4. the explicit-nuisance model uses the full objective and retains prior/auxiliary reweighting;
5. corrected generation, absolute evidence, posterior prediction, and selection integrals exercise the additional Table-2 capabilities.

Before using figures in a paper:

- run full mode with `LOAD_IF_AVAILABLE=False` after this v4 architecture/loss change;
- inspect memberwise flow NLLs, scalar inversion checks, joint-tail Wasserstein/quantile audits, and mixture density fingerprints;
- report conditional $\log Z$, raw and fully normalized bridge variance, ESS, Pareto-$k$, maximum weight, and head saturation;
- compare all three ablations with independent seeds, not only the committed seed;
- vary every importance candidate-pool size in generation/evidence/predictive applications;
- keep the pre-selection caveat with the selection-integral figure;
- interpret bridge improvement as internal consistency only; simulator closure and calibration remain decisive;
- use standalone scripts under `exercise9_multiclass_v4_figures_scripts/full/`.

The independently trained binary construction in the original Exercise 9 remains the key external baseline.
